In [1]:
%pip install torch transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 라이브러리 불러오기

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
# BertTokenizer : 문장을 BERT가 이해할 수 있는 숫자 형태로 바꿔주는 도구
# BertForSequenceClassification: 문장 분류용 BERT 모델입니다. -> BERT 뒤에 분류기가 붙어 있는 모델
from torch.optim import AdamW #Bert에서 많이 쓰이는 Optimization
from sklearn.model_selection import train_test_split

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 데이터 준비

In [2]:
texts = [
    "this movie is great",
    "i really loved this film",
    "amazing acting and wonderful story",
    "this was fantastic",
    "i enjoyed the whole movie",
    "this movie is terrible",
    "i hated this film",
    "boring and disappointing",
    "this was so bad",
    "waste of time"
]

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# 입력 문장 -> 클래스 1개

### 학습용 / 검증용 데이터 나누기

In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

### 토크나이저 준비

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# BERT는 문장을 그대로 읽지 못함 -> 반드시 숫자 형태의 입력으로 바꿔야 함.
# bert는 tokenizer는 wordpiece tokenizer를 사용

토큰화 예시

```
"this movie is great"
```

토큰화 후 개념적으로

```
["[CLS]", "this","movie","is","great","[SEP]"]
```
(CLS - clasfficiation, SEP - Separator)

그리고 이것이 숫자로 바뀜

예를 들어

[101, 023, 3185, 2003, 2307, 102]

실제 숫자는 모델마다 다를 수 있음.




### 토크나이저가 실제로 무엇을 만드는지 보기

In [5]:
# 한 문장을 직접 토큰화할 수 있음.

sample = tokenizer(
    "this movie is great",
    add_special_tokens=True, # BERT의 필요한 특수 토큰 [CLS], [SEP]를 붙임
    max_length=10,           # 문장의 최대 길이는 10으로 맞춤
    padding="max_length",    # 길이가 짧으면 뒤에 0을 넣어 길이를 맞춤.
    truncation=True,         # 문장이 너무 길면 잘라냄.
    return_tensors="pt"      # 결과를 PyTorch 텐서로 반환
)

print(sample)

{'input_ids': tensor([[ 101, 2023, 3185, 2003, 2307,  102,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}


- input_ids: 토큰들의 숫자 ID
- attention_mask: 실제 단어 위치는 1, 패딩 위치는 0

## DataSet 만들기

문장 1개씩 꺼내서 BERT 입력 형태로 바꾸는 클래스를 만듦

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts          # 문장들
        self.labels = labels        # 정답들
        self.tokenizer = tokenizer  # BERT 토크나이저
        self.max_len = max_len      # 최대 문장 길이

    def __len__(self):              # 데이터 개수 반환
        return len(self.texts)

    def __getitem__(self, idx):     #idx번째 문장을 꺼내서 BERT 입력으로 바꾸고,
        text = self.texts[idx]      # 정답라벨도 같이 반환
        label = self.labels[idx]

        encoding = self.tokenizer(  # 문장 하나를 BERT 입력으로 변환.
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            # Dataset에서 문장 하나를 반환할 때는 (max_len,) 형태가 더 편하므로
            # 앞의 1차원을 없애기 위해 squeeze(0)을 사용
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
            # 분류 문제의 라벨은 정수형이여야 하고, Pytorch에서 보통 long 타입을 사용
        }
        return item

### Dataset 객체 만들기

In [ ]:
train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

### DataLoader 만들기

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)

- DataLoader: Dataset에서 데이터를 꺼내서 배치 단위로 묶어줌
    - batch_size=2: 문장 2개식 한 번에 모델에 넣겠다.
    - shuffle=True: 학습할 때 데이터를 섞음. 보통 train에서는 섞고, validation에서는 섞지 않음.

- 딥러닝 모델은 보통 한 문장씩보다 여러 문장을 묶어서 처리하는 것이 효율적
    - 문장 1개 -> 너무 느림
    - 문장 여러 개 -> 효율적

### 모델 불러오기

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# 아래: BERT 본체
# 위: 분류용 layer로 구성됨

## num_labels=2 -> 클래스가 2개(0:부정, 1:긍정)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### GPU 또는 CPU 설정


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# 입력 데이터도 같은 device에 보내야 함(cpu or gpu)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Optimizer 설정

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)
# BERT 파인튜닝에서는 보통 학습률을 작게 둠(ex.2e-5,3e-5,5e-5)

### 학습 함수 만들기

In [ ]:
def train_one_epoch(model, data_loader, optimizer, device):
    model.train()    # 모델을 학습 모드로 바꿈(Dropout 같은 층이 학습 모드로 동작)
    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:   # DataLoader에서 배치 단위로 데이터를 가져옴.(한 번에 문장 2개씩 가져오게 됨)
        input_ids = batch["input_ids"].to(device) # 모델과 데이터가 같은 장치에 있어야 함.
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad() #gradient 초기화 (이전 배치의 gradient 초기화)
        # Pytorch는 gradient를 누적하므로 매 배치마다 꼭 초기화해줘야 함.

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss      # 모델이 얼마나 틀렸는지 나타내는 값(작을수록 좋음)
        logits = outputs.logits  # 각 클래스에 대한 점수

        loss.backward()          # loss를 gradient를 계산함.
        optimizer.step()         # 계산된 gradient를 이용해 모델 파라미터를 업데이트함.

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1) # 각 문장에서 가장 점수가 큰 클래스를 선택함.
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy
    # 예측과 정답이 같은 개수를 세고, 전체 개수로 나누어 accuracy를 구함.

In [ ]:
def eval_model(model, data_loader, device):
    model.eval() # 평가 모드로 전환
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad(): # 평가할 때는 gradient가 필요 없으므로 계산하지 않음.
        #메모리도 절약되고 속도도 조금 좋아짐.
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy

### 학습 진행

In [ ]:
epochs = 3

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print("-" * 50)

Epoch 1/3
Train Loss: 0.7109, Train Acc: 0.5000
Val   Loss: 0.7297, Val   Acc: 0.5000
--------------------------------------------------
Epoch 2/3
Train Loss: 0.6261, Train Acc: 0.7500
Val   Loss: 0.7136, Val   Acc: 0.5000
--------------------------------------------------
Epoch 3/3
Train Loss: 0.5019, Train Acc: 0.8750
Val   Loss: 0.7021, Val   Acc: 0.5000
--------------------------------------------------


### 새 문장 예측 함수 만들기

In [ ]:
def predict_sentiment(text, model, tokenizer, device, max_len=32):
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

    return pred

## 실제 예측해보기

In [ ]:
test_text = "this film was really wonderful"
pred = predict_sentiment(test_text, model, tokenizer, device)

print("입력 문장:", test_text)
print("예측 라벨:", pred)
# 1이면 긍정, 0이면 부정

입력 문장: this film was really wonderful
예측 라벨: 1


- 한국어 BERT

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

texts = [
    "이 영화 정말 재미있어요",
    "연기가 훌륭하고 스토리도 좋았어요",
    "정말 감동적인 작품이었어요",
    "배우들의 연기가 인상적이었어요",
    "이 영화는 최고예요",
    "이 영화는 별로였어요",
    "내용이 너무 지루했어요",
    "시간이 아까운 영화였어요",
    "스토리가 엉성하고 재미없었어요",
    "다시는 보고 싶지 않아요"
]

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# train / validation 분리
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# 한국어 BERT 모델 / 토크나이저 불러오기
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

#  Dataset 만들기
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }
        return item

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

#  DataLoader 만들기
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


optimizer = AdamW(model.parameters(), lr=2e-5)

def train_one_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy


def eval_model(model, data_loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy

epochs = 3

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print("-" * 50)

def predict_sentiment(text, model, tokenizer, device, max_len=32):
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

    return pred

# 새 문장 예측
test_text = "정말 재미있고 감동적인 영화였어요"
pred = predict_sentiment(test_text, model, tokenizer, device)

print("입력 문장:", test_text)
print("예측 라벨:", pred)   # 1: 긍정, 0: 부정

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

Epoch 1/3
Train Loss: 0.6894, Train Acc: 0.6250
Val   Loss: 0.5620, Val   Acc: 1.0000
--------------------------------------------------
Epoch 2/3
Train Loss: 0.4959, Train Acc: 0.8750
Val   Loss: 0.4317, Val   Acc: 1.0000
--------------------------------------------------
Epoch 3/3
Train Loss: 0.3232, Train Acc: 1.0000
Val   Loss: 0.3337, Val   Acc: 1.0000
--------------------------------------------------
입력 문장: 정말 재미있고 감동적인 영화였어요
예측 라벨: 1


## KOBERT

- KoBERT는 SKTBrain의 한국어 특화 BERT
- 공개 저장소에는 12개 레이어, hidden size 768, 12개 attention head, 최대 길이 512, 그리고 SentencePiece 기반 8,002 크기 vocabulary

In [ ]:
#Kobert저장소는 SentencePiece 기반 토크나이저를 사용
##KOBERT는 vocabulary도 한글 위키 기반 SentencePiece tokenizer로 학습되어 설명
%pip install torch transformers scikit-learn sentencepiece

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
# Dataset, Dataloder: 배치 학습용 데이터 처리
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# AutoTokenizer: KoBERT 토크나이저 불러오기
#AutoModelForSequenceClassification: 문장 분류용 KoBERT 모델
from torch.optim import AdamW
# BERT계열의 파인튜닝에서 많이 쓰이는 optimizer
from sklearn.model_selection import train_test_split

### 예제 데이터 준비

In [ ]:
texts = [
    "이 영화 정말 재미있어요",
    "배우들 연기가 훌륭했어요",
    "스토리가 감동적이었어요",
    "정말 만족스러운 작품이었어요",
    "시간 가는 줄 모르고 봤어요",
    "이 영화는 너무 지루했어요",
    "내용이 엉성하고 별로였어요",
    "다시는 보고 싶지 않아요",
    "시간이 아까운 영화였어요",
    "연출이 실망스러웠어요"
]

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

### 학습용 / 검증용 데이터 나누기

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

### KOBERT 토크나이저와 모델 불러오기

In [ ]:
model_name = "skt/kobert-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# 문장을 KoBERT 입력용 숫자로 변환
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
# model: KoBERT + 문장분류 head

config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: skt/kobert-base-v1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 토크나이저가 무엇을 만드는지 먼저 보자

In [ ]:
sample = tokenizer(
    "이 영화 정말 재미있어요",
    max_length=16,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print(sample["input_ids"])      # input_ids: 토큰 번호
print(sample["attention_mask"]) # attention_mask: 실제 토큰은 1, 패딩은 0

tensor([[517, 491, 494, 517, 491,   0, 493,   0, 517,   0, 517,   0, 494, 491,
           3,   2]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


### Dataset 만들기

In [ ]:
 # 문장 하나를 꺼낼 때마다 자동으로 토큰화되도록 Dataset을 만듦
class KoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0), # 결과가 (1,max_len)이라서 앞의 1차원을 제거
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long) # 분류 라벨은 정수형 long으로 두는 것이 안전
        }
        return item

### Dataset 객체 만들기

In [ ]:
train_dataset = KoBERTDataset(train_texts, train_labels, tokenizer)
val_dataset = KoBERTDataset(val_texts, val_labels, tokenizer)

### DataLoader 만들기

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)
# batch_size=2: 한 번에 문장 2개씩 학습
# shuffle=True: train 데이터는 섞어서 학습

In [ ]:
# CPU or GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# KoBERT 저장소의 fine-tuning에서도 GPU 사용을 권장

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(8002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

In [ ]:
# Optimizer 설정
optimizer = AdamW(model.parameters(), lr=2e-5)
# KoBert 계열이므로 학습률을 보통 작게 둠.

In [ ]:
# epoch 함수 만들기
def train_one_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# 평가 함수 만들기
def train_one_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# 학습 실행
epochs = 3

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print("-" * 50)

Epoch 1/3
Train Loss: 0.6777, Train Acc: 0.6250
Val   Loss: 0.6952, Val   Acc: 0.5000
--------------------------------------------------
Epoch 2/3
Train Loss: 0.7441, Train Acc: 0.1250
Val   Loss: 0.6912, Val   Acc: 1.0000
--------------------------------------------------
Epoch 3/3
Train Loss: 0.7046, Train Acc: 0.2500
Val   Loss: 0.6960, Val   Acc: 0.5000
--------------------------------------------------


In [ ]:
def predict_sentiment(text, model, tokenizer, device, max_len=32):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        score = probs[0][pred].item()

    label_map = {0: "부정", 1: "긍정"}

    return { 
        "text": text,
        "pred_id": pred,
        "pred_label": label_map[pred],
        "confidence": round(score, 4)
    }

In [ ]:
# 예측 실행
result = predict_sentiment(
    "배우 연기도 좋고 정말 재미있었어요",
    model,
    tokenizer,
    device
)

print(result)

{'text': '배우 연기도 좋고 정말 재미있었어요', 'pred_id': 1, 'pred_label': '긍정', 'confidence': 0.5454}


## CSV 파일을 읽어서 KoBERT로 실제 데이터 학습

In [ ]:
import pandas as pd
import random

random.seed(42)

positive_subjects = ["이 영화", "이 작품", "이 드라마", "이번 영화", "이 콘텐츠"]
negative_subjects = ["이 영화", "이 작품", "이 드라마", "이번 영화", "이 콘텐츠"]

positive_reasons = [
    "스토리가 탄탄해서",
    "배우들 연기가 좋아서",
    "연출이 섬세해서",
    "감정선이 잘 살아 있어서",
    "몰입감이 높아서",
    "전개가 흥미로워서"
]

negative_reasons = [
    "스토리가 약해서",
    "전개가 느려서",
    "연출이 어색해서",
    "몰입이 잘 안 돼서",
    "기대보다 별로여서",
    "내용이 산만해서"
]

positive_results = [
    "정말 재미있었어요",
    "매우 만족스러웠어요",
    "감동적으로 봤어요",
    "다시 보고 싶어요",
    "추천하고 싶어요",
    "완성도가 높다고 느꼈어요"
]

negative_results = [
    "너무 지루했어요",
    "정말 실망스러웠어요",
    "끝까지 보기 힘들었어요",
    "시간이 아까웠어요",
    "추천하고 싶지 않아요",
    "기대에 못 미쳤어요"
]

templates = [
    "{subject}는 {reason} {result}",
    "{reason} {subject}는 {result}",
    "{subject}는 전체적으로 {result}",
    "{subject}는 보고 나서 {result}"
]

def make_sentences(subjects, reasons, results, templates, n):
    sentences = []
    for _ in range(n):
        subject = random.choice(subjects)
        reason = random.choice(reasons)
        result = random.choice(results)
        template = random.choice(templates)
        sentence = template.format(subject=subject, reason=reason, result=result)
        sentences.append(sentence)
    return sentences

positive_texts = make_sentences(
    positive_subjects, positive_reasons, positive_results, templates, 300
)

negative_texts = make_sentences(
    negative_subjects, negative_reasons, negative_results, templates, 300
)

texts = positive_texts + negative_texts
labels = [1] * len(positive_texts) + [0] * len(negative_texts)

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df = df.drop_duplicates().sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv("train.csv", index=False, encoding="utf-8-sig")

print(df.head(10))
print("총 데이터 개수:", len(df))
print("파일 저장 완료: train.csv")

                                text  label
0         이 콘텐츠는 전개가 흥미로워서 감동적으로 봤어요      1
1  감정선이 잘 살아 있어서 이 작품는 완성도가 높다고 느꼈어요      1
2          이 작품는 연출이 어색해서 기대에 못 미쳤어요      0
3            이번 영화는 전체적으로 매우 만족스러웠어요      1
4             이 영화는 전체적으로 기대에 못 미쳤어요      0
5       몰입감이 높아서 이 작품는 완성도가 높다고 느꼈어요      1
6        이 영화는 몰입이 잘 안 돼서 기대에 못 미쳤어요      0
7       이 콘텐츠는 연출이 어색해서 끝까지 보기 힘들었어요      0
8        이 작품는 내용이 산만해서 끝까지 보기 힘들었어요      0
9              이 영화는 보고 나서 정말 재미있었어요      1
총 데이터 개수: 336
파일 저장 완료: train.csv


In [ ]:
%pip install torch transformers pandas scikit-learn sentencepiece

In [ ]:
pip install torch transformers pandas scikit-learn sentencepiece

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv", encoding='utf-8-sig')
# utf-8-sig: 이 파일은 UTF-8로 작성되었습니다"
print(df.head())
print(df.columns)
print(df.shape)

                                text  label
0         이 콘텐츠는 전개가 흥미로워서 감동적으로 봤어요      1
1  감정선이 잘 살아 있어서 이 작품는 완성도가 높다고 느꼈어요      1
2          이 작품는 연출이 어색해서 기대에 못 미쳤어요      0
3            이번 영화는 전체적으로 매우 만족스러웠어요      1
4             이 영화는 전체적으로 기대에 못 미쳤어요      0
Index(['text', 'label'], dtype='object')
(336, 2)


In [ ]:
# 필요한 열만 남기기
df = df[["text", "label"]].dropna()

print(df.head())
print(df.columns)
print(df.shape)

                                text  label
0         이 콘텐츠는 전개가 흥미로워서 감동적으로 봤어요      1
1  감정선이 잘 살아 있어서 이 작품는 완성도가 높다고 느꼈어요      1
2          이 작품는 연출이 어색해서 기대에 못 미쳤어요      0
3            이번 영화는 전체적으로 매우 만족스러웠어요      1
4             이 영화는 전체적으로 기대에 못 미쳤어요      0
Index(['text', 'label'], dtype='object')
(336, 2)


In [ ]:
texts = df["text"].astype(str).tolist()
labels = df["label"].astype(int).tolist()

print(texts[:3])
print(labels[:3])

                                text  label
0         이 콘텐츠는 전개가 흥미로워서 감동적으로 봤어요      1
1  감정선이 잘 살아 있어서 이 작품는 완성도가 높다고 느꼈어요      1
2          이 작품는 연출이 어색해서 기대에 못 미쳤어요      0
3            이번 영화는 전체적으로 매우 만족스러웠어요      1
4             이 영화는 전체적으로 기대에 못 미쳤어요      0
Index(['text', 'label'], dtype='object')
(336, 2)


In [ ]:
# 텍스트와 라벨을 리스트로 만들기
texts = df["text"].astype(str).tolist()   # texts: 문장들만 모은 리스트, .astype: 문장을 문자열로 맞춤
labels = df["label"].astype(int).tolist() # lables: 정답들만 모은 리스트, .astype: 라벨을 정수로 맞춤

print(texts[:3])
print(labels[:3])

['이 콘텐츠는 전개가 흥미로워서 감동적으로 봤어요', '감정선이 잘 살아 있어서 이 작품는 완성도가 높다고 느꼈어요', '이 작품는 연출이 어색해서 기대에 못 미쳤어요']
[1, 1, 0]


In [ ]:
# 학습용과 검증용으로 나누기
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("학습 데이터 개수:", len(train_texts))
print("검증 데이터 개수:", len(val_texts))

학습 데이터 개수: 268
검증 데이터 개수: 68


In [ ]:
# KoBERT 토크나이저와 모델 불러오기
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "skt/kobert-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)       # 문장을 KoBERT가 이해하는 숫자 형태로 바꿈
model = AutoModelForSequenceClassification.from_pretrained( # KoBERT 본체 + 분류기
    model_name,
    num_labels=2 # 클래스가 2개 -> 0: 부정, 1: 긍정
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: skt/kobert-base-v1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
sample = tokenizer(
    "이 영화는 정말 재미있었어요",
    max_length=16,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print(sample["input_ids"])      # 단어를 숫자로 바꾼 결과
print(sample["attention_mask"]) # 실제 단어는 1, 패딩은 0

tensor([[517, 491, 494, 517, 491,   0, 493,   0, 517,   0, 517,   0, 494, 491,
           3,   2]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
# Dataset 만들기
## 문장을 하나씩 꺼내면 자동으로 토큰화되도록 Dataset을 만듦.
from torch.utils.data import Dataset

class KoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Dataset 객체 만들기
train_dataset = KoBERTDataset(train_texts, train_labels, tokenizer, max_len=64)
val_dataset = KoBERTDataset(val_texts, val_labels, tokenizer, max_len=64)

In [ ]:
# DataLoader 만들기
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(device)

cpu


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
# 학습 함수 만들기
def train_one_epoch(model, data_loader, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    acc = correct / total

    return avg_loss, acc

In [ ]:
# 평가 함수 만들기

def eval_model(model, data_loader, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    acc = correct / total

    return avg_loss, acc


In [ ]:
# 학습 진행
epochs = 3

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

Epoch 1/3
Train Loss: 0.7145 | Train Acc: 0.4925
Val   Loss: 0.6952 | Val   Acc: 0.4853
--------------------------------------------------
Epoch 2/3
Train Loss: 0.6923 | Train Acc: 0.5448
Val   Loss: 0.6985 | Val   Acc: 0.5147
--------------------------------------------------
Epoch 3/3
Train Loss: 0.7058 | Train Acc: 0.4627
Val   Loss: 0.6923 | Val   Acc: 0.5147
--------------------------------------------------


In [ ]:
# 모델 저장하기
save_path = "./kobert_sentiment_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("모델 저장 완료")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료


In [ ]:
# 예측 함수 만들기
def predict_text(text, model, tokenizer, device, max_len=64):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)

        pred_id = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_id].item()

    label_map = {
        0: "부정",
        1: "긍정"
    }

    return {
        "text": text,
        "pred_id": pred_id,
        "pred_label": label_map[pred_id],
        "confidence": round(confidence, 4)
    }

In [ ]:
# 실제 예측하기
print(predict_text("배우들 연기도 좋고 스토리도 감동적이었어요", model, tokenizer, device))
print(predict_text("너무 지루해서 끝까지 보기 힘들었어요", model, tokenizer, device))
print(predict_text("전체적으로는 괜찮았지만 조금 아쉬웠어요", model, tokenizer, device))

## 1. RoBERTa란?

RoBERTa는 **BERT를 더 잘 학습시키기 위해 제안된 모델**이다.  
이름은 **Robustly Optimized BERT Pretraining Approach**의 약자.

RoBERTa는 BERT의 기본 아이디어를 유지하면서도, **학습 데이터, 학습 방법, 하이퍼파라미터를 더 최적화한 모델**이라고 볼 수 있다.

한 문장으로 말하면:

> **RoBERTa는 BERT의 구조를 크게 바꾸기보다, pretraining 과정을 개선하여 성능을 높인 모델이다.**

---

## 2. 왜 RoBERTa가 등장했는가?

BERT 이후 많은 후속 모델들이 발표되었는데, RoBERTa 논문은 다음과 같은 질문을 던진다.

> "정말 새로운 구조가 필요했던 것일까?"  
> "아니면 BERT를 충분히 잘 학습시키지 않았던 것일까?"

RoBERTa는 후자에 주목하였다.  

**BERT의 한계가 구조 때문만은 아니며, 학습 방식 자체를 더 잘 설계하면 성능이 크게 좋아질 수 있다**고 본 것이다.

---

## 3. 핵심 아이디어

RoBERTa의 핵심은 다음과 같다.

- **더 큰 데이터셋 사용**
- **더 긴 학습**
- **더 큰 batch size 사용**
- **Next Sentence Prediction(NSP) 제거**
- **Dynamic Masking 적용**
- **Byte-level BPE tokenizer 사용**

구조 혁신보다는 **학습 최적화**가 중심이다.

---

## 4. BERT와 RoBERTa의 차이

## 4.1 NSP 제거

BERT는 두 가지 pretraining task를 사용하였다.

1. **MLM (Masked Language Modeling)**  
   일부 단어를 가리고 맞히는 작업

2. **NSP (Next Sentence Prediction)**  
   두 문장이 실제로 이어지는 문장쌍인지 예측하는 작업

RoBERTa는 이 중에서 **NSP를 제거**하였다.

### 이유
RoBERTa 논문은 NSP가 반드시 성능 향상에 중요한 것은 아니라고 보았다.  
오히려 **MLM에 더 집중하고 전체 학습을 더 풍부하게 하는 것이 효과적**이라고 판단하였다.

### 의미
RoBERTa는 문장 관계를 억지로 분류하는 보조 작업보다 **언어 전체의 문맥 표현을 더 깊이 학습하는 데 집중**한 것.

---

## 4.2 더 큰 데이터셋

RoBERTa는 BERT보다 더 큰 텍스트 데이터로 학습.

BERT는 비교적 제한된 데이터로 학습되었지만, RoBERTa는 훨씬 더 방대한 corpus를 사용하여 pretraining을 수행.

### 의미
딥러닝 모델은 구조도 중요하지만,  
**얼마나 많은 데이터를 충분히 학습했는가**가 매우 중요.

RoBERTa는 이를 잘 보여주는 사례.

---

## 4.3 더 긴 학습과 큰 batch size

RoBERTa는 BERT보다:

- 더 오래 학습하고
- 더 큰 mini-batch를 사용하고
- 더 적절한 하이퍼파라미터를 적용하였다

### 의미
BERT의 원래 결과는 모델 구조의 한계라기보다  
**학습이 충분히 최적화되지 않았기 때문일 수 있다**는 점을 보여줌.

RoBERTa는 **"BERT를 더 제대로 학습시킨 버전"**이라고 이해해도 좋다.

---

## 4.4 Dynamic Masking

BERT는 일반적으로 한 번 정해진 마스킹 패턴을 기준으로 학습하는 방식으로 이해할 수 있다.  
반면 RoBERTa는 **dynamic masking**을 사용한다.

### Dynamic Masking이란?
같은 문장이 여러 번 학습에 사용되더라도  
매번 **다른 위치를 마스킹**할 수 있게 하는 방식이다.

예를 들어 문장이 다음과 같다고 하자.

- `The cat sits on the mat.`

한 번은 `cat`을 가리고,  
다른 번은 `sits`를 가리고,  
또 다른 번은 `mat`을 가릴 수 있다.

### 장점
- 모델이 특정 마스크 패턴에만 익숙해지는 것을 방지
- 더 다양한 복원 문제를 경험
- 일반화 성능 향상 가능

---

## 4.5 Tokenizer 차이

BERT는 주로 **WordPiece tokenizer**를 사용한다.  
RoBERTa는 **byte-level BPE tokenizer**를 사용한다.

### 의미
이 차이는 다음에 영향을 준다.

- 토큰 분할 방식
- 공백 처리
- 희귀 단어 처리
- 입력 인코딩 방식

따라서 **BERT용 tokenizer와 RoBERTa용 tokenizer는 서로 그대로 바꿔 쓰면 안 된다.**

---

## 5. RoBERTa의 구조

RoBERTa의 기본 구조는 여전히 **Transformer Encoder**이다.

다음 요소들은 BERT와 거의 같다.

- Multi-Head Self-Attention
- Feed Forward Network
- Residual Connection
- Layer Normalization
- Positional Embedding

따라서 RoBERTa를 이해할 때 중요한 점은 **구조적 혁신보다는 학습 전략의 변화**라는 것.

---

## 6. 입력 방식

RoBERTa도 텍스트를 토큰으로 나눈 뒤 embedding하여 Transformer Encoder에 입력한다.

기본 흐름은 다음과 같다.

1. 문장을 tokenizer로 분할
2. 각 토큰을 임베딩 벡터로 변환
3. positional embedding 추가
4. Transformer Encoder에 통과
5. 각 토큰의 문맥 표현(contextual representation) 획득

BERT와 마찬가지로 RoBERTa도 **양방향 문맥 정보**를 활용하여 문장을 이해함.

---

## 7. RoBERTa의 장점

## 7.1 BERT보다 높은 성능
RoBERTa는 여러 NLP benchmark에서 BERT보다 더 좋은 성능을 보였다.

대표적으로 다음과 같은 작업에서 강점을 보였다.

- 문장 분류
- 감성 분석
- 자연어 추론
- 질문응답
- 독해

---

## 7.2 구조 변경 없이 성능 향상
RoBERTa의 가장 큰 의미 중 하나는  
**복잡한 새 구조 없이도 학습 최적화만으로 큰 성능 향상이 가능하다**는 점을 보여준 것.

이는 연구적으로도 매우 중요하다.

---

## 7.3 BERT 계열 모델 이해에 좋은 기준점
RoBERTa는 BERT 이후 모델들을 공부할 때 좋은 기준이 된다.

예를 들어:

- 구조를 바꾼 모델인지
- 학습 목표를 바꾼 모델인지
- 학습 레시피를 개선한 모델인지

이런 구분을 할 때 RoBERTa는 대표적인 **학습 최적화형 모델**이다.

---

## 8. RoBERTa의 한계

RoBERTa가 강력한 모델이긴 하지만, 한계도 있다.

### 8.1 연산량이 큼
더 큰 데이터, 더 긴 학습, 더 큰 batch를 사용하므로  
학습 비용이 크다.

### 8.2 생성 모델은 아님
RoBERTa는 BERT처럼 **encoder-only 모델**이므로  
GPT처럼 자연스러운 텍스트를 순차적으로 생성하는 데 최적화된 구조는 아니다.

### 8.3 문장 생성보다 이해 tasks에 적합
RoBERTa는 특히 다음과 같은 **이해 중심 task**에 적합하다.

- classification
- tagging
- sentence similarity
- extractive QA

---

## 9. BERT와 RoBERTa 비교 요약

| 항목 | BERT | RoBERTa |
|---|---|---|
| 기본 구조 | Transformer Encoder | Transformer Encoder |
| 주요 목표 | MLM + NSP | MLM |
| NSP 사용 | 사용 | 제거 |
| Masking | 상대적으로 고정된 방식 | Dynamic Masking |
| Tokenizer | WordPiece | Byte-level BPE |
| 학습 데이터 | 상대적으로 적음 | 더 큼 |
| 학습 길이 | 기본 설정 | 더 오래 학습 |
| Batch size | 비교적 작음 | 더 큼 |
| 핵심 특징 | 양방향 문맥 학습 | BERT 학습 최적화 |

---

## 10. 한 줄 요약

> **RoBERTa는 BERT의 구조를 크게 바꾸지 않고, pretraining 과정을 더 강력하게 최적화하여 성능을 높인 모델이다.**

# RoBERTa의 수식 정리

## 1. 전체 개요

RoBERTa는 기본적으로 **Transformer Encoder**를 사용하며,  
사전학습(pretraining)에서는 **Masked Language Modeling (MLM)** 만 사용한다.

즉, 핵심은 다음 두 부분이다.

1. **입력 문장을 embedding하여 Transformer Encoder에 넣는 과정**
2. **가려진 토큰(masked token)을 예측하는 MLM loss**

---

## 2. 입력 표현(Input Representation)

입력 문장을 토큰 시퀀스로 나타내면 다음과 같다.

$$
x = (x_1, x_2, \dots, x_n)
$$

각 토큰 $x_i$는 임베딩 벡터로 변환.

RoBERTa의 입력 벡터는 보통 다음처럼 생각할 수 있다.

$$
h_i^{(0)} = E(x_i) + P_i
$$

여기서

- $E(x_i)$: 토큰 $x_i$의 token embedding
- $P_i$: 위치 $i$에 대한 positional embedding
- $h_i^{(0)}$: encoder의 첫 입력 벡터

각 위치의 입력은 **토큰 의미 정보 + 위치 정보**의 합이다.

전체 입력 행렬로 쓰면

$$
H^{(0)} = [h_1^{(0)}, h_2^{(0)}, \dots, h_n^{(0)}]
$$

---

## 3. Self-Attention

Transformer Encoder의 핵심은 self-attention이다.

각 층에서 입력 $H$에 대해 Query, Key, Value를 만든다.

$$
Q = HW_Q,\quad K = HW_K,\quad V = HW_V
$$

여기서

- $H \in \mathbb{R}^{n \times d}$: 입력 hidden states
- $W_Q, W_K, W_V \in \mathbb{R}^{d \times d_k}$: 학습 파라미터
- $(Q, K, V$): query, key, value 행렬

---

## 4. Scaled Dot-Product Attention

attention score는 query와 key의 내적으로 계산한다.

$$
\text{score}(Q,K) = QK^\top
$$

하지만 차원이 커지면 값이 너무 커질 수 있으므로, 보통 $\sqrt{d_k}$로 나누어 스케일링한다.

$$
\text{Attention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

이 수식이 self-attention의 핵심이다.

### 의미
- $QK^\top$: 각 토큰이 다른 토큰을 얼마나 참고할지 계산
- softmax: 각 토큰에 대한 attention weight 생성
- $V$: 그 가중치를 실제 정보(value)에 적용

각 토큰은 다른 토큰들을 가중합하여 새로운 문맥 표현을 얻게 된다.

---

## 5. Multi-Head Attention

Transformer는 attention을 한 번만 하지 않고, 여러 head로 나누어 수행한다.

각 head에 대해

$$
\text{head}_j
=
\text{Attention}(Q_j, K_j, V_j)
$$

이고, 이들을 이어붙여서(concatenate) 다시 선형변환한다.

$$
\text{MultiHead}(H)
=
\text{Concat}(\text{head}_1,\dots,\text{head}_m)W_O
$$

여기서

- $m$: attention head 개수
- $W_O$: 출력 projection matrix

### 의미
각 head가 서로 다른 관계를 학습하여 문장의 다양한 문맥 정보를 동시에 반영할 수 있다.

---

## 6. Add & Norm

attention 결과는 residual connection과 layer normalization을 거친다.

먼저 attention block 출력이 $A$라고 하면,

$$
\tilde{H} = \text{LayerNorm}(H + A)
$$

그 다음 feed-forward network를 통과시킨 뒤 다시 residual connection을 적용.

$$
H' = \text{LayerNorm}(\tilde{H} + \text{FFN}(\tilde{H}))
$$

한 encoder block은 대략 다음 흐름.

$$
H
\rightarrow
\text{MultiHeadAttention}
\rightarrow
\text{Add\&Norm}
\rightarrow
\text{FFN}
\rightarrow
\text{Add\&Norm}
$$

---

## 7. Feed-Forward Network

각 위치별로 동일한 MLP를 적용한다.

$$
\text{FFN}(x) = W_2 \,\sigma(W_1 x + b_1) + b_2
$$

여기서

- $W_1, W_2$: 가중치 행렬
- $b_1, b_2$: bias
- $\sigma$: 활성화 함수 (보통 GELU)

RoBERTa에서도 BERT와 마찬가지로 보통 GELU가 사용된다.

$$
\text{GELU}(x) = x \Phi(x)
$$

여기서 $\Phi(x)$는 표준정규분포의 누적분포함수이다.

실제로는 근사식도 자주 쓴다.

$$
\text{GELU}(x)
\approx
0.5x\left(1+\tanh\left(\sqrt{\frac{2}{\pi}}\left(x+0.044715x^3\right)\right)\right)
$$

---

## 8. Encoder 출력

Transformer Encoder를 \(L\)층 통과하면 최종 표현은

$$
H^{(L)} = (h_1^{(L)}, h_2^{(L)}, \dots, h_n^{(L)})
$$

이 된다.

각 $h_i^{(L)}$는단순히 $x_i$ 하나만의 표현이 아니라, **문장 전체 문맥을 반영한 contextual embedding**.

---

## 9. Masked Language Modeling (MLM)

RoBERTa의 사전학습 목표는 MLM 하나이다.

입력 토큰들 중 일부 위치 집합을 \(M\)이라 하자.  

$M$은 mask된 위치들의 집합이다.

예를 들어 문장이

$$
\text{“The cat sits on the mat”}
$$

일 때, `cat`과 `mat`을 가렸다면 그 위치들이 $M$에 해당.

RoBERTa는 각 masked position $i \in M$에 대해 원래 토큰 $x_i$를 맞히도록 학습.

각 위치 $i$에서 최종 hidden state $h_i^{(L)}$를 사용하여  
어휘 집합 $Vocab$ 위의 확률분포를 만든다.

$$
p(x_i \mid x_{\setminus M})
=
\text{softmax}(W h_i^{(L)} + b)
$$

여기서

- $x_{\setminus M}$: mask되지 않은 나머지 토큰들
- $W, b$: 출력층 파라미터
- $p(x_i \mid x_{\setminus M})$: 가려진 위치 $i$의 원래 토큰 확률

---

## 10. MLM Loss

RoBERTa는 mask된 위치에 대해서만 loss를 계산.

$$
\mathcal{L}_{MLM}
=
-\sum_{i \in M} \log p(x_i \mid x_{\setminus M})
$$

평균으로 쓰면

$$
\mathcal{L}_{MLM}
=
-\frac{1}{|M|}
\sum_{i \in M} \log p(x_i \mid x_{\setminus M})
$$

### 의미
- 정답 토큰의 확률이 높아지도록 학습
- mask되지 않은 위치는 직접 loss에 포함되지 않음
- 문맥을 바탕으로 가려진 단어를 복원하는 능력을 학습

이것이 RoBERTa pretraining의 핵심 목적함수.

---

## 11. BERT와의 수식 차이: NSP 제거

BERT는 MLM 외에 NSP(Next Sentence Prediction) loss도 사용.


BERT의 전체 loss는 보통

$$
\mathcal{L}_{BERT}
=
\mathcal{L}_{MLM}
+
\mathcal{L}_{NSP}
$$

형태로 생각할 수 있다.

반면 RoBERTa는 NSP를 제거했으므로

$$
\mathcal{L}_{RoBERTa}
=
\mathcal{L}_{MLM}
$$

만 사용.

이 점이 수식적으로 가장 큰 차이이다.

---

## 12. Dynamic Masking의 수학적 의미

RoBERTa는 training 중 매 epoch 또는 반복에서 mask 집합 $M$을 고정하지 않고 바꿀 수 있다.

같은 입력 \(x\)에 대해서도 매번 다른 \(M\)을 샘플링한다.

$$
M \sim \mathcal{D}_{mask}
$$

그리고 그때마다 loss는

$$
\mathcal{L}_{MLM}(x, M)
=
-\sum_{i \in M} \log p(x_i \mid x_{\setminus M})
$$

로 달라진다.

### 의미
같은 문장을 반복해서 보더라도 항상 같은 위치를 맞히는 것이 아니라  
여러 위치를 다양하게 복원하게 되어 일반화에 도움이 된다.

In [ ]:
#fill mask 예측(마스크된 위치의 단어를 예측)

# 설치
# %pip install transformers torch

from transformers import pipeline

# RoBERTa 기반 fill-mask 파이프라인
unmasker = pipeline("fill-mask", model="FacebookAI/roberta-base")

text = "Paris is the <mask> of France."
results = unmasker(text)

for r in results:
    print(f"token: {r['token_str']}, score: {r['score']:.4f}, sequence: {r['sequence']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

token:  capital, score: 0.8638, sequence: Paris is the capital of France.
token:  heart, score: 0.0556, sequence: Paris is the heart of France.
token:  Capital, score: 0.0276, sequence: Paris is the Capital of France.
token:  center, score: 0.0155, sequence: Paris is the center of France.
token:  city, score: 0.0038, sequence: Paris is the city of France.


In [ ]:
#직접 tokenizer와 model로 예측
# %pip install transformers torch

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

text = "Paris is the <mask> of France."

# 토큰화
inputs = tokenizer(text, return_tensors="pt")

# 모델 추론
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits   # (batch, seq_len, vocab_size)

# <mask> 위치 찾기
mask_token_id = tokenizer.mask_token_id
mask_position = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=True)[1]

# 해당 위치의 vocab logits
mask_logits = logits[0, mask_position, :]

# 상위 5개 토큰
top_k = 5
top_tokens = torch.topk(mask_logits, top_k, dim=-1).indices[0].tolist()

print("Top predictions:")
for token_id in top_tokens:
    token = tokenizer.decode([token_id]).strip()
    print(token)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top predictions:
capital
heart
Capital
center
city


In [ ]:
#예측 확률까지 보기
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

text = "The man worked as a <mask>."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
mask_pos = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
mask_logits = logits[0, mask_pos, :]
probs = torch.softmax(mask_logits, dim=-1)

top_k = 10
top_probs, top_ids = torch.topk(probs, top_k, dim=-1)

print("Top predictions with probabilities:\n")
for prob, idx in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode([idx]).strip()
    print(f"{token:15s}  {prob.item():.4f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top predictions with probabilities:

mechanic         0.0870
waiter           0.0820
butcher          0.0733
miner            0.0463
guard            0.0402
cleaner          0.0380
clerk            0.0273
bartender        0.0257
cook             0.0235
courier          0.0235


In [ ]:
#토큰화 결과 확인
## RoBERTa는 byte-level BPE tokenizer를 사용 / BERT와 다르게 쓰임.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

text = "I am learning RoBERTa today."
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", ids)

Tokens: ['I', 'Ġam', 'Ġlearning', 'ĠRo', 'BER', 'Ta', 'Ġtoday', '.']
Token IDs: [100, 524, 2239, 3830, 11126, 38495, 452, 4]


In [ ]:
#문장 임베딩 비슷하게 hidden state 보기
##RoBERTa는 encoder 모델이므로, 문장 전체에 대한 contextual representation을 얻을 수 있습니다.

# %pip install transformers torch

import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")
model = AutoModel.from_pretrained("FacebookAI/roberta-base")

text = "RoBERTa is a strong encoder model."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state

print("Hidden state shape:", last_hidden_state.shape)
# (batch_size, seq_len, hidden_size)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden state shape: torch.Size([1, 12, 768])


In [ ]:
#아주 간단한 분류 실습 틀
## RoBERTa는 fill-mask뿐 아니라 sequence classification, token classification, question answering 등에 fine-tuning해서 많이 씀.
# %pip install transformers datasets evaluate accelerate torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np

# 예시 데이터
data = {
    "text": [
        "I love this movie.",
        "This film was terrible.",
        "The acting was excellent.",
        "I do not like this story."
    ],
    "label": [1, 0, 1, 0]
}

dataset = Dataset.from_dict(data)

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=32)

encoded_dataset = dataset.map(preprocess)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./roberta-cls",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset
)

trainer.train()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,0.702469
2,0.705692
3,0.649776
4,0.674345
5,0.778744
6,0.679464


TrainOutput(global_step=6, training_loss=0.6984152396519979, metrics={'train_runtime': 1.1381, 'train_samples_per_second': 10.544, 'train_steps_per_second': 5.272, 'total_flos': 197333291520.0, 'train_loss': 0.6984152396519979, 'epoch': 3.0})

In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask", model="FacebookAI/roberta-base")

results = unmasker("The capital of France is <mask>.")

for r in results[:5]:
    print(r["sequence"], r["score"])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The capital of France is Paris. 0.9036234021186829
The capital of France is Lyon. 0.080291748046875
The capital of France is Nice. 0.004803333431482315
The capital of France is Nancy. 0.0020990855991840363
The capital of France is Napoleon. 0.0011298992903903127


In [ ]:
# %pip install -q transformers torch

from transformers import pipeline
import pandas as pd

# 1. 모델 준비
bert_fill = pipeline("fill-mask", model="google-bert/bert-base-uncased")
roberta_fill = pipeline("fill-mask", model="FacebookAI/roberta-base")

# 2. 비교용 문장들
# BERT는 [MASK], RoBERTa는 <mask> 를 사용해야 하므로
# 같은 문장을 각각 다른 마스크 토큰으로 넣어줍니다.
test_sentences = [
    {
        "name": "capital",
        "bert_text": "The capital of France is [MASK].",
        "roberta_text": "The capital of France is <mask>."
    },
    {
        "name": "profession",
        "bert_text": "The man worked as a [MASK].",
        "roberta_text": "The man worked as a <mask>."
    },
    {
        "name": "science",
        "bert_text": "The experiment was repeated to improve [MASK].",
        "roberta_text": "The experiment was repeated to improve <mask>."
    },
    {
        "name": "context",
        "bert_text": "After months of drought, the rain was a great [MASK].",
        "roberta_text": "After months of drought, the rain was a great <mask>."
    },
    {
        "name": "medical",
        "bert_text": "The doctor reviewed the patient's [MASK] before surgery.",
        "roberta_text": "The doctor reviewed the patient's <mask> before surgery."
    },
    {
        "name": "logic",
        "bert_text": "Because the evidence was weak, the claim remained [MASK].",
        "roberta_text": "Because the evidence was weak, the claim remained <mask>."
    },
]

def get_topk(pipe, text, k=5):
    results = pipe(text, top_k=k)
    cleaned = []
    for r in results:
        cleaned.append({
            "token_str": r["token_str"].strip(),
            "score": float(r["score"]),
            "sequence": r["sequence"]
        })
    return cleaned

# 3. 결과 수집
rows = []

for item in test_sentences:
    bert_results = get_topk(bert_fill, item["bert_text"], k=5)
    roberta_results = get_topk(roberta_fill, item["roberta_text"], k=5)

    for rank in range(5):
        rows.append({
            "case": item["name"],
            "rank": rank + 1,
            "BERT_token": bert_results[rank]["token_str"],
            "BERT_score": bert_results[rank]["score"],
            "RoBERTa_token": roberta_results[rank]["token_str"],
            "RoBERTa_score": roberta_results[rank]["score"]
        })

df = pd.DataFrame(rows)
print(df)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


          case  rank     BERT_token  BERT_score RoBERTa_token  RoBERTa_score
0      capital     1          paris    0.416788         Paris       0.903623
1      capital     2          lille    0.071417          Lyon       0.080292
2      capital     3           lyon    0.063393          Nice       0.004803
3      capital     4      marseille    0.044447         Nancy       0.002099
4      capital     5          tours    0.030297      Napoleon       0.001130
5   profession     1      carpenter    0.097476      mechanic       0.087024
6   profession     2         waiter    0.052383        waiter       0.081965
7   profession     3         barber    0.049627       butcher       0.073323
8   profession     4       mechanic    0.037886         miner       0.046322
9   profession     5       salesman    0.037681         guard       0.040150
10     science     1        results    0.741965       results       0.330330
11     science     2    performance    0.067067   sensitivity       0.116187

In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
roberta_tok = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

text = "The patient's preoperative assessment was incomplete."

print("BERT tokens:")
print(bert_tok.tokenize(text))
print()

print("RoBERTa tokens:")
print(roberta_tok.tokenize(text))

BERT tokens:
['the', 'patient', "'", 's', 'pre', '##oper', '##ative', 'assessment', 'was', 'incomplete', '.']

RoBERTa tokens:
['The', 'Ġpatient', "'s", 'Ġpre', 'operative', 'Ġassessment', 'Ġwas', 'Ġincomplete', '.']


- 한국어

In [ ]:
from transformers import pipeline
import pandas as pd

# 한국어 KLUE BERT, RoBERTa
bert_fill = pipeline("fill-mask", model="klue/bert-base")
roberta_fill = pipeline("fill-mask", model="klue/roberta-base")

# 비교할 문장들
# 두 모델 모두 [MASK] 토큰으로 테스트하는 것이 편합니다.
test_sentences = [
    "대한민국의 수도는 [MASK]이다.",
    "의사는 수술 전에 환자의 [MASK]을 확인했다.",
    "오랜 가뭄 끝에 내린 비는 정말 큰 [MASK]이었다.",
    "실험의 재현성을 높이기 위해 [MASK] 측정을 반복했다.",
    "증거가 충분하지 않아서 그 주장은 아직 [MASK] 상태로 남아 있다.",
    "이 논문의 핵심 [MASK]은 새로운 최적화 방법을 제안한 것이다."
]

def get_topk(pipe, text, k=5):
    results = pipe(text, top_k=k)
    cleaned = []
    for r in results:
        cleaned.append({
            "token_str": r["token_str"].strip(),
            "score": float(r["score"]),
            "sequence": r["sequence"]
        })
    return cleaned

rows = []

for sent in test_sentences:
    bert_results = get_topk(bert_fill, sent, k=5)
    roberta_results = get_topk(roberta_fill, sent, k=5)

    for rank in range(5):
        rows.append({
            "문장": sent,
            "순위": rank + 1,
            "BERT 예측": bert_results[rank]["token_str"],
            "BERT 점수": bert_results[rank]["score"],
            "RoBERTa 예측": roberta_results[rank]["token_str"],
            "RoBERTa 점수": roberta_results[rank]["score"]
        })

df = pd.DataFrame(rows)
print(df)

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

...

RobertaForMaskedLM LOAD REPORT from: klue/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

                                         문장  순위 BERT 예측   BERT 점수 RoBERTa 예측  \
0                       대한민국의 수도는 [MASK]이다.   1      서울  0.595462         서울   
1                       대한민국의 수도는 [MASK]이다.   2     광화문  0.081063         부산   
2                       대한민국의 수도는 [MASK]이다.   3      평양  0.047202      서울특별시   
3                       대한민국의 수도는 [MASK]이다.   4      부산  0.029445       대한민국   
4                       대한민국의 수도는 [MASK]이다.   5      인천  0.026465         대전   
5               의사는 수술 전에 환자의 [MASK]을 확인했다.   1      맥박  0.065162         건강   
6               의사는 수술 전에 환자의 [MASK]을 확인했다.   2      건강  0.055786         병력   
7               의사는 수술 전에 환자의 [MASK]을 확인했다.   3      혈액  0.048859          병   
8               의사는 수술 전에 환자의 [MASK]을 확인했다.   4      의식  0.042780         시력   
9               의사는 수술 전에 환자의 [MASK]을 확인했다.   5      증상  0.033373         사인   
10           오랜 가뭄 끝에 내린 비는 정말 큰 [MASK]이었다.   1      축복  0.528181         축복   
11           오랜 가뭄 끝에 내린 비는 정말 큰 [MASK]이

In [ ]:
# 문장별로 출력
for sent in test_sentences:
    print("=" * 90)
    print("입력 문장:", sent)
    print()

    bert_results = get_topk(bert_fill, sent, k=5)
    roberta_results = get_topk(roberta_fill, sent, k=5)

    print("[KLUE BERT top-5]")
    for i, r in enumerate(bert_results, 1):
        print(f"{i}. {r['token_str']:12s}  score={r['score']:.4f}")

    print()
    print("[KLUE RoBERTa top-5]")
    for i, r in enumerate(roberta_results, 1):
        print(f"{i}. {r['token_str']:12s}  score={r['score']:.4f}")
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


입력 문장: 대한민국의 수도는 [MASK]이다.

[KLUE BERT top-5]
1. 서울            score=0.5955
2. 광화문           score=0.0811
3. 평양            score=0.0472
4. 부산            score=0.0294
5. 인천            score=0.0265

[KLUE RoBERTa top-5]
1. 서울            score=0.7482
2. 부산            score=0.0442
3. 서울특별시         score=0.0421
4. 대한민국          score=0.0234
5. 대전            score=0.0219

입력 문장: 의사는 수술 전에 환자의 [MASK]을 확인했다.

[KLUE BERT top-5]
1. 맥박            score=0.0652
2. 건강            score=0.0558
3. 혈액            score=0.0489
4. 의식            score=0.0428
5. 증상            score=0.0334

[KLUE RoBERTa top-5]
1. 건강            score=0.0475
2. 병력            score=0.0422
3. 병             score=0.0341
4. 시력            score=0.0227
5. 사인            score=0.0197

입력 문장: 오랜 가뭄 끝에 내린 비는 정말 큰 [MASK]이었다.

[KLUE BERT top-5]
1. 축복            score=0.5282
2. 재앙            score=0.0655
3. 충격            score=0.0463
4. 선물            score=0.0313
5. 고통            score=0.0289

[KLUE RoBERTa top-5]
1. 축복            score=0.

In [ ]:
# 더 차이가 나는 문장
hard_cases = [
    "연구 결과가 유의미했지만 표본 수가 적어서 아직 [MASK] 단계에 머물러 있다.",
    "위원회는 근거가 불충분하다고 판단하여 그 제안을 [MASK]했다.",
    "백신의 단기 효과는 확인되었지만 장기적인 영향은 아직 [MASK]이다.",
    "증인의 진술은 매우 [MASK]이어서 사건의 흐름을 이해하는 데 도움이 되었다.",
    "새로운 알고리즘은 계산 비용을 줄이면서도 높은 [MASK]를 유지했다."
]

for sent in hard_cases:
    print("=" * 90)
    print("입력 문장:", sent)
    print()

    bert_results = get_topk(bert_fill, sent, k=5)
    roberta_results = get_topk(roberta_fill, sent, k=5)

    print("[KLUE BERT top-5]")
    for i, r in enumerate(bert_results, 1):
        print(f"{i}. {r['token_str']:12s}  score={r['score']:.4f}")

    print()
    print("[KLUE RoBERTa top-5]")
    for i, r in enumerate(roberta_results, 1):
        print(f"{i}. {r['token_str']:12s}  score={r['score']:.4f}")
    print()

입력 문장: 연구 결과가 유의미했지만 표본 수가 적어서 아직 [MASK] 단계에 머물러 있다.

[KLUE BERT top-5]
1. 조사            score=0.2368
2. 초기            score=0.2175
3. 연구            score=0.1199
4. 실험            score=0.0514
5. 논의            score=0.0241

[KLUE RoBERTa top-5]
1. 초기            score=0.2979
2. 실험            score=0.2016
3. 연구            score=0.1323
4. 마무리           score=0.0350
5. 중간            score=0.0282

입력 문장: 위원회는 근거가 불충분하다고 판단하여 그 제안을 [MASK]했다.

[KLUE BERT top-5]
1. 철회            score=0.1669
2. 다시            score=0.1319
3. 취소            score=0.0712
4. 받아들여야         score=0.0680
5. 기각            score=0.0668

[KLUE RoBERTa top-5]
1. 기각            score=0.3149
2. 철회            score=0.1811
3. 반려            score=0.0949
4. 각하            score=0.0787
5. 취소            score=0.0639

입력 문장: 백신의 단기 효과는 확인되었지만 장기적인 영향은 아직 [MASK]이다.

[KLUE BERT top-5]
1. 미지수           score=0.9619
2. 의문            score=0.0171
3. 시기상조          score=0.0033
4. ##도           score=0.0016
5. 미스터리          score=0.0010

[K

In [ ]:
# 정답 후보 예측 하는 함수
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

def target_token_score(model_name, text, target_word):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)

    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    mask_id = tokenizer.mask_token_id
    mask_pos = (inputs["input_ids"] == mask_id).nonzero(as_tuple=True)[1].item()

    target_ids = tokenizer.encode(target_word, add_special_tokens=False)

    # 한 토큰으로 매핑되는 경우만 단순 비교
    if len(target_ids) != 1:
        return None

    probs = torch.softmax(logits[0, mask_pos], dim=-1)
    return float(probs[target_ids[0]])

text = "대한민국의 수도는 [MASK]이다."

bert_score = target_token_score("klue/bert-base", text, "서울")
roberta_score = target_token_score("klue/roberta-base", text, "서울")

print("KLUE BERT    - '서울' 점수:", bert_score)
print("KLUE RoBERTa - '서울' 점수:", roberta_score)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: klue/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KLUE BERT    - '서울' 점수: 0.595461905002594
KLUE RoBERTa - '서울' 점수: 0.7481499910354614


In [ ]:
# 토크나이저 차이
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("klue/bert-base")
roberta_tok = AutoTokenizer.from_pretrained("klue/roberta-base")

text = "이 논문의 핵심 기여는 새로운 최적화 기법에 있다."

print("[KLUE BERT 토큰화]")
print(bert_tok.tokenize(text))
print()

print("[KLUE RoBERTa 토큰화]")
print(roberta_tok.tokenize(text))

[KLUE BERT 토큰화]
['이', '논문', '##의', '핵심', '기여', '##는', '새로운', '최적화', '기법', '##에', '있', '##다', '.']

[KLUE RoBERTa 토큰화]
['이', '논문', '##의', '핵심', '기여', '##는', '새로운', '최적화', '기법', '##에', '있', '##다', '.']


## 1. ELECTRA란?

ELECTRA는 **BERT 계열의 사전학습 언어모델**이지만,  BERT처럼 마스크된 토큰을 복원하는 방식만 쓰지 않고,  **바꿔치기된 토큰을 구별하는 replaced token detection (RTD)** 을 핵심 학습 목표로 사용하는 모델.

논문 제목은 다음과 같다.

> **ELECTRA: Pre-training Text Encoders as Discriminators Rather Than Generators**

한 문장으로 정리하면,

> **ELECTRA는 “빈칸을 맞히는 모델”이 아니라, “이 토큰이 원래 것이냐, 바뀐 것이냐”를 판별하도록 학습하는 모델이다.**

---

## 2. 왜 ELECTRA가 등장했는가?

BERT의 MLM(Masked Language Modeling)은 강력하지만,  
실제로는 **전체 토큰 중 일부 마스크된 위치에 대해서만 직접 학습 신호가 들어간다**는 한계가 있다.


- 입력 문장 전체를 사용하더라도
- loss는 주로 마스크된 일부 위치에 대해서만 계산된다.

ELECTRA는 이 비효율을 줄이기 위해 제안되었다.

핵심 질문은 다음과 같다.

> **왜 일부 마스크 위치만 학습하지?**  
> **왜 문장 전체 토큰을 더 적극적으로 활용하지 않지?**

ELECTRA는 여기에 대해,

- 작은 **generator**가 일부 토큰을 그럴듯하게 바꾸고
- **discriminator**가 각 토큰이 원래 것인지, 교체된 것인지 판별하게 하자

는 방식으로 답한다.

---

## 3. 핵심 아이디어

ELECTRA는 두 모델을 함께 사용한다.

### 3.1 Generator
- 작은 MLM 모델
- 일부 마스크된 위치에 대해 대체 토큰을 생성

### 3.2 Discriminator
- 문장 전체 각 토큰에 대해
- 그 토큰이 **원래 토큰인지(original)**,
- 아니면 **generator가 바꿔 넣은 토큰인지(replaced)** 판별

BERT가

> “이 빈칸에 들어갈 단어는 무엇인가?”

를 배우는 모델이라면,

ELECTRA는

> “이 자리에 있는 단어가 진짜 원래 단어인가, 아니면 바꿔치기된 단어인가?”

를 배우는 모델.

---

## 4. 학습 과정

ELECTRA의 사전학습은 대략 다음과 같이 진행된다.

### 4.1 원본 문장 준비

예를 들어 입력 문장이 다음과 같다고 하자.

```text
The cat sits on the mat.

### 4.2 일부 토큰 마스킹

예를 들어 cat, mat을 마스크 하면 다음과 같다.

```text
The cat sits on the mat.


### 4.3 Generator가 대체 토큰 생성

Generator는 MLM처럼 마스크된 위치에 예측한다.

예를 들어 다음처럼 생성할 수 있다.

```text
The dog sits on the floor.
```

Generator가 넣는 단어가 완전히 엉뚱한 것이 아니라 문맥상 그럴듯한 단어인 경우가 많음.

### 4.4 Discriminator가 각 토큰 판별

Discriminator는 문장의 각 위치에 대해 다음을 예측.

- original
- replaced

예를 들면:

- The → original
- dog → replaced
- sits → original
- on → original
- the → original
- floor → replaced

문장 전체 토큰마다 이진 분류(binary classification) 를 수행.

## 5. BERT와의 차이
###5.1 학습 목표 차이
**BERT**
- MLM: 마스크된 단어 맞히기
- NSP: 다음 문장 예측(원 논문 기준)

**ELECTRA**

- Generator: MLM 수행
- Discriminator: RTD 수행

가장 큰 차이는 최종 표현을 학습시키는 핵심 목표가 다르다는 점.

- BERT는 복원 문제
- ELECTRA는 판별 문제

를 중심으로 학습.

### 5.2 학습 신호의 밀도

BERT는 마스크된 일부 위치에서만 직접 loss를 계산.

반면 ELECTRA의 discriminator는 모든 토큰 위치에 대해 loss를 계산할 수 있다.



- BERT: sparse supervision
- ELECTRA: dense supervision

이라고 볼 수 있다.

이 점 때문에 ELECTRA는 같은 계산량에서도 더 효율적으로 학습될 수 있다.

5.3 실제로 사용하는 모델

ELECTRA는 학습할 때는 generator와 discriminator를 함께 사용하지만,
실제 downstream task에서는 보통 discriminator만 사용한다.

즉,

학습 시: generator + discriminator
사용 시: discriminator

이다.

## 6. 왜 ELECTRA가 좋은가?
### 6.1 거의 모든 토큰에서 학습 가능

BERT는 보통 마스크된 일부 위치만 직접 학습.

하지만 ELECTRA는 문장 전체의 거의 모든 위치에서
“이 토큰이 진짜인가?”를 학습.

따라서 같은 입력을 사용하더라도
더 많은 supervision을 받게 됨.

###6.2 더 어려운 문맥 판별 문제를 학습

Generator가 넣는 토큰은 아주 그럴듯한 단어일 수 있다.

예를 들어,

- cat → dog
- mat → floor

처럼 바뀐다면 문장이 겉보기에는 꽤 자연스럽다.

그래서 discriminator는 단순히 어색함만 보는 것이 아니라,
더 정교한 문맥 이해를 해야 한다.

### 6.3 계산 효율이 좋음

ELECTRA는 같은 데이터와 비슷한 계산 자원에서
MLM 기반 모델보다 더 효율적으로 학습될 수 있다는 점이 큰 장점.

특히 작은 모델에서 이 장점이 잘 드러난다.

## 7. 수식으로 이해하기
##7.1 입력 시퀀스

입력 문장을 다음과 같이 두자.
$$
x = (x_1,x_2, \cdots, x_n)
$$

여기서 $x_i$는 $i$번째 토큰

###7.2 Generator의 MLM Loss

일부 마스크 위치의 집합을 $M$이라 하자.

Generator는 BERT처럼 마스크된 토큰을 예측
$$
\mathcal{L}_G = -\sum_{i \in M} \log p_G(x_i \mid x_{\setminus M})
$$

- $P_G$: Generator의 예측 확률
- $x_i \mid x_{\setminus M}$: 마스크되지 않은 나머지 문맥

### 7.3 Replaced Input 만들기

Generator가 각 마스크 위치에 대해 대체 토큰을 샘플링해서  
새로운 문장 \(\tilde{x}\)를 만든다.

$$
\tilde{x} = (\tilde{x}_1, \tilde{x}_2, \dots, \tilde{x}_n)
$$

각 위치 $i$에 대해 라벨 $y_i$를 정의하면,

$$
y_i =
\begin{cases}
1, & \text{if } \tilde{x}_i = x_i \\
0, & \text{if } \tilde{x}_i \neq x_i
\end{cases}
$$


- 원래 단어면 $1$
- 교체된 단어면 $0$

## 7.4 Discriminator의 RTD

Discriminator는 각 위치에 대해  
그 토큰이 원래 토큰인지 예측.

$$
D(\tilde{x}, i) = P(y_i = 1 \mid \tilde{x})
$$

그리고 전체 loss는 각 위치에 대한 이진 분류 손실.

$$
\mathcal{L}_{D}
=
-\sum_{i=1}^{n}
\left[
y_i \log D(\tilde{x}, i)
+
(1-y_i)\log(1-D(\tilde{x}, i))
\right]
$$

이 식이 ELECTRA의 핵심.

ELECTRA는

> **문장 전체의 각 토큰에 대해 original / replaced를 구분하는 binary classification 문제**

를 푸는 것.

## 7.5 전체 손실 함수

전체 손실은 보통 다음처럼 쓸 수 있다.

$$
\mathcal{L} = \mathcal{L}_{D} + \lambda \mathcal{L}_{G}
$$

여기서 $\lambda$는 generator loss와 discriminator loss의 균형을 조절하는 가중치.

하지만 실제 핵심은 **discriminator 학습**이며,  
downstream task에서 사용하는 것도 보통 discriminator이다.


## 8. 구조는 Transformer인가?

ELECTRA도 기본적으로는 **Transformer encoder 기반 모델**.

즉, 다음 구조는 BERT와 거의 같다.

- Multi-Head Self-Attention
- Feed Forward Network
- Residual Connection
- Layer Normalization
- Positional Embedding

따라서 ELECTRA의 핵심 차이는 **구조보다는 사전학습 목표 함수**에 있다.

## 9. ELECTRA의 장점

### 9.1 작은 모델에서도 강함

ELECTRA는 특히 작은 모델에서도 좋은 성능을 보이는 것으로 알려져 있다.  
즉, 적은 계산량으로도 강한 표현을 배울 수 있다.

### 9.2 계산 대비 효율이 좋음

같은 compute budget에서  
BERT류 MLM 모델보다 더 효율적으로 학습할 수 있다는 점이 중요한 장점.

### 9.3 이해 중심 과제에 적합

ELECTRA는 encoder 기반 모델이므로 다음과 같은 task에 잘 맞는다.

- 문장 분류
- 감성 분석
- 자연어 추론
- 개체명 인식
- 토큰 분류
- 질의응답

즉, **text generation**보다는 **text understanding**에 더 적합.


## 10. ELECTRA의 한계

### 10.1 설명이 BERT보다 복잡함

BERT는 MLM으로 비교적 쉽게 설명할 수 있지만,  
ELECTRA는 generator와 discriminator 두 모델의 상호작용을 이해해야 하므로  
개념적으로 조금 더 복잡.

### 10.2 Generator 품질의 영향

Generator가 너무 약하면 너무 이상한 토큰을 넣게 되어  
discriminator가 쉽게 맞힐 수 있다.

반대로 generator가 너무 강하면  
학습 비용이 커지거나 전체 구조가 무거워질 수 있다.

즉, 두 모델의 균형이 중요.

### 10.3 생성 모델은 아님

ELECTRA는 GPT처럼 문장을 순차적으로 생성하는 모델이 아니다.  
본질적으로 **판별형 encoder 모델**이다.

따라서 자유 생성보다는  
문장 이해, 분류, 판별 task에 더 적합.


## 11. BERT, RoBERTa, ELECTRA 비교

| 항목 | BERT | RoBERTa | ELECTRA |
|---|---|---|---|
| 기본 구조 | Transformer Encoder | Transformer Encoder | Transformer Encoder |
| 핵심 학습 목표 | MLM + NSP | MLM | RTD + 보조적 MLM |
| 마스크 복원 | 예 | 예 | generator에서만 예 |
| 판별 학습 | 아니오 | 아니오 | 예 |
| 학습 신호 | 주로 마스크 위치 | 주로 마스크 위치 | 거의 모든 위치 |
| 핵심 장점 | 양방향 문맥 학습 | BERT 학습 최적화 | 계산 효율성과 높은 sample efficiency |

이 표의 핵심은 다음과 같다.

- **BERT**: 기본적인 bidirectional encoder
- **RoBERTa**: BERT를 더 잘 학습시킨 버전
- **ELECTRA**: 학습 목표 자체를 바꿔서 효율을 높인 버전

#ELECTRA 실습
- BERT/RoBERTa는 빈칸 채우기 모델
- ELECTRA는 진짜/가짜 토큰 판별 모델
- 그래서 ELECTRA는 출력 자체가 “예측 단어”가 아니라 “각 토큰의 suspicious score라는 점

In [ ]:
# 비교용 문장 준비
## "Paris"는 맞는 토큰
## "Lyon"은 문법적으로 자연스럽지만 사실 틀림
## "banana"는 문맥상 매우 이상

sentences = [
    "The capital of France is Paris.",
    "The capital of France is Lyon.",
    "The doctor reviewed the patient's chart before surgery.",
    "The doctor reviewed the patient's banana before surgery.",
    "The cat sits on the mat.",
    "The cat sits on the floor.",
]

In [ ]:
# BERT나 RoBERT는 [MASK] 또는 <MASK> 위치의 단어를 예측
## BERT는 masked token prediction과 NSP로 사전 학습되었고,
## RoBert는 BERT를 기반으로 NSP를 제거하고 학습 설정을 최적화한 모델

from transformers import pipeline

bert_fill = pipeline("fill-mask", model="google-bert/bert-base-uncased")
roberta_fill = pipeline("fill-mask", model="FacebookAI/roberta-base")

text_bert = "The capital of France is [MASK]."
text_roberta = "The capital of France is <mask>."

print("[BERT]")
bert_results = bert_fill(text_bert, top_k=5)
for r in bert_results:
    print(r["token_str"].strip(), round(r["score"], 4), " | ", r["sequence"])

print("\n[RoBERTa]")
roberta_results = roberta_fill(text_roberta, top_k=5)
for r in roberta_results:
    print(r["token_str"].strip(), round(r["score"], 4), " | ", r["sequence"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[BERT]
paris 0.4168  |  the capital of france is paris.
lille 0.0714  |  the capital of france is lille.
lyon 0.0634  |  the capital of france is lyon.
marseille 0.0444  |  the capital of france is marseille.
tours 0.0303  |  the capital of france is tours.

[RoBERTa]
Paris 0.9036  |  The capital of France is Paris.
Lyon 0.0803  |  The capital of France is Lyon.
Nice 0.0048  |  The capital of France is Nice.
Nancy 0.0021  |  The capital of France is Nancy.
Napoleon 0.0011  |  The capital of France is Napoleon.


- 무슨 단어가 올까? 예측을 하고 ELECRTA는 여기서 다르게 감.

### ELECTRA 핵심 실습: discriminaotr로 "진짜/가짜"보기

Hugging Face의 ELECTRA 문서는 generator와 discriminator를 모두 다룰 수 있지만, 실제 핵심 모델은 discriminator라고 설명합니다. 또한 generator만 MLM으로 학습되며, discriminator는 교체 여부를 판별

In [ ]:
#모델 카드는 small ELECTRA 모델과 single-GPU pretraining/fine-tuning 코드를 포함한다고 안내
import torch
from transformers import AutoTokenizer, ElectraForPreTraining

model_name = "google/electra-small-discriminator"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = ElectraForPreTraining.from_pretrained(model_name)
model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/54.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ElectraForPreTraining(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_fea

In [ ]:
# 각 토큰이 "가짜일 확률" 보기
##ElectraForPreTraining의 출력은 각 토큰 위치마다 하나의 logit.
##이 값에 sigmoid를 취하면, 그 토큰이 replaced/fake일 가능성을 볼 수 있습니다.

def electra_token_scores(text):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0]              # (seq_len,)
    probs_fake = torch.sigmoid(logits)      # 각 토큰이 "fake"일 확률처럼 해석

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    results = []
    for tok, score in zip(tokens, probs_fake):
        results.append((tok, float(score)))
    return results


In [ ]:
# 정상 문장 VS 이상 문장 비교

test_1 = "The doctor reviewed the patient's chart before surgery."
test_2 = "The doctor reviewed the patient's banana before surgery."

for text in [test_1, test_2]:
    print("=" * 80)
    print("INPUT:", text)
    print("-" * 80)
    scores = electra_token_scores(text)
    for tok, score in scores:
        print(f"{tok:15s}  fake_score={score:.4f}")
    print()

INPUT: The doctor reviewed the patient's chart before surgery.
--------------------------------------------------------------------------------
[CLS]            fake_score=0.0000
the              fake_score=0.0202
doctor           fake_score=0.0883
reviewed         fake_score=0.0849
the              fake_score=0.0229
patient          fake_score=0.0786
'                fake_score=0.0373
s                fake_score=0.0007
chart            fake_score=0.0859
before           fake_score=0.0747
surgery          fake_score=0.0538
.                fake_score=0.0001
[SEP]            fake_score=0.0000

INPUT: The doctor reviewed the patient's banana before surgery.
--------------------------------------------------------------------------------
[CLS]            fake_score=0.0000
the              fake_score=0.0225
doctor           fake_score=0.0755
reviewed         fake_score=0.1000
the              fake_score=0.0214
patient          fake_score=0.0679
'                fake_score=0.0438
s         

In [ ]:
def show_electra_scores(text, threshold=0.5):
    scores = electra_token_scores(text)
    print("=" * 100)
    print("INPUT:", text)
    print("=" * 100)
    print(f"{'TOKEN':20s} {'FAKE_SCORE':>12s} {'FLAG':>10s}")
    print("-" * 100)

    for tok, score in scores:
        flag = "fake?" if score >= threshold else ""
        print(f"{tok:20s} {score:12.4f} {flag:>10s}")

show_electra_scores("The capital of France is Paris.")
show_electra_scores("The capital of France is Lyon.")
show_electra_scores("The doctor reviewed the patient's banana before surgery.")

INPUT: The capital of France is Paris.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000           
the                        0.0029           
capital                    0.0130           
of                         0.0046           
france                     0.0159           
is                         0.0048           
paris                      0.0528           
.                          0.0011           
[SEP]                      0.0000           
INPUT: The capital of France is Lyon.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0001           
the                        0.0048           
capital                    0.0078           
of                         0.0065           
france                     0.0060           
i

In [ ]:
test_1 = "The doctor reviewed the patient's chart before surgery."
test_2 = "The doctor reviewed the patient's banana before surgery."

for text in [test_1, test_2]:
    print("=" * 80)
    print("INPUT:", text)
    print("-" * 80)
    scores = electra_token_scores(text)
    for tok, score in scores:
        print(f"{tok:15s}  fake_score={score:.4f}")
    print()

INPUT: The doctor reviewed the patient's chart before surgery.
--------------------------------------------------------------------------------
[CLS]            fake_score=0.0000
the              fake_score=0.0202
doctor           fake_score=0.0883
reviewed         fake_score=0.0849
the              fake_score=0.0229
patient          fake_score=0.0786
'                fake_score=0.0373
s                fake_score=0.0007
chart            fake_score=0.0859
before           fake_score=0.0747
surgery          fake_score=0.0538
.                fake_score=0.0001
[SEP]            fake_score=0.0000

INPUT: The doctor reviewed the patient's banana before surgery.
--------------------------------------------------------------------------------
[CLS]            fake_score=0.0000
the              fake_score=0.0225
doctor           fake_score=0.0755
reviewed         fake_score=0.1000
the              fake_score=0.0214
patient          fake_score=0.0679
'                fake_score=0.0438
s         

- generator와 discriminator를 같이 보자.

    - generator: “이 자리에 어떤 단어를 넣을까?”
    - discriminator: “지금 들어간 단어가 가짜인가?”

In [ ]:
from transformers import ElectraForMaskedLM

gen_name = "google/electra-small-generator"
disc_name = "google/electra-small-discriminator"

gen_tokenizer = AutoTokenizer.from_pretrained(gen_name)
generator = ElectraForMaskedLM.from_pretrained(gen_name)
discriminator = ElectraForPreTraining.from_pretrained(disc_name)

generator.eval()
discriminator.eval()

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie electra.embeddings.word_embeddings.weight to generator_lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ElectraForPreTraining(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_fea

In [ ]:
#generator가 가짜 토큰 만들고, discriminator가 잡아내기

def electra_full_demo(masked_text):
    print("=" * 100)
    print("원본 입력:", masked_text)

    # 1) generator로 [MASK] 채우기
    inputs = gen_tokenizer(masked_text, return_tensors="pt")
    mask_id = gen_tokenizer.mask_token_id
    mask_positions = (inputs["input_ids"][0] == mask_id).nonzero(as_tuple=True)[0]

    with torch.no_grad():
        gen_outputs = generator(**inputs)

    gen_logits = gen_outputs.logits[0]  # (seq_len, vocab_size)

    replaced_ids = inputs["input_ids"][0].clone()

    for pos in mask_positions:
        pred_id = gen_logits[pos].argmax().item()
        replaced_ids[pos] = pred_id

    replaced_text = gen_tokenizer.decode(replaced_ids, skip_special_tokens=True)
    print("generator가 채운 문장:", replaced_text)

    # 2) discriminator로 각 토큰 fake score 확인
    disc_inputs = {"input_ids": replaced_ids.unsqueeze(0), "attention_mask": inputs["attention_mask"]}

    with torch.no_grad():
        disc_outputs = discriminator(**disc_inputs)

    disc_logits = disc_outputs.logits[0]
    disc_probs = torch.sigmoid(disc_logits)
    disc_tokens = gen_tokenizer.convert_ids_to_tokens(replaced_ids)

    print("-" * 100)
    print(f"{'TOKEN':20s} {'FAKE_SCORE':>12s}")
    print("-" * 100)
    for tok, score in zip(disc_tokens, disc_probs):
        print(f"{tok:20s} {float(score):12.4f}")

In [ ]:
electra_full_demo("The capital of France is [MASK].")
electra_full_demo("The doctor reviewed the patient's [MASK] before surgery.")

원본 입력: The capital of France is [MASK].
generator가 채운 문장: the capital of france is paris.
----------------------------------------------------------------------------------------------------
TOKEN                  FAKE_SCORE
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000
the                        0.0029
capital                    0.0130
of                         0.0046
france                     0.0159
is                         0.0048
paris                      0.0528
.                          0.0011
[SEP]                      0.0000
원본 입력: The doctor reviewed the patient's [MASK] before surgery.
generator가 채운 문장: the doctor reviewed the patient ' s condition before surgery.
----------------------------------------------------------------------------------------------------
TOKEN                  FAKE_SCORE
-----------------------------------------------------------------------------------------

In [ ]:
#BERT / RoBERTa / ELECTRA 차이를 코드로 나란히 비교

def compare_models():
    print("=" * 100)
    print("1) BERT / RoBERTa: 빈칸 채우기")
    print("=" * 100)

    bert_text = "The doctor reviewed the patient's [MASK] before surgery."
    roberta_text = "The doctor reviewed the patient's <mask> before surgery."

    print("[BERT top-5]")
    for r in bert_fill(bert_text, top_k=5):
        print(r["token_str"].strip(), round(r["score"], 4))

    print("\n[RoBERTa top-5]")
    for r in roberta_fill(roberta_text, top_k=5):
        print(r["token_str"].strip(), round(r["score"], 4))

    print("\n" + "=" * 100)
    print("2) ELECTRA: 현재 토큰이 가짜인지 판별")
    print("=" * 100)

    show_electra_scores("The doctor reviewed the patient's chart before surgery.")
    show_electra_scores("The doctor reviewed the patient's banana before surgery.")

compare_models()

1) BERT / RoBERTa: 빈칸 채우기
[BERT top-5]
condition 0.4213
symptoms 0.0822
appearance 0.0489
behavior 0.0296
health 0.0278

[RoBERTa top-5]
history 0.1808
records 0.1306
chart 0.1057
record 0.0426
notes 0.0256

2) ELECTRA: 현재 토큰이 가짜인지 판별
INPUT: The doctor reviewed the patient's chart before surgery.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000           
the                        0.0202           
doctor                     0.0883           
reviewed                   0.0849           
the                        0.0229           
patient                    0.0786           
'                          0.0373           
s                          0.0007           
chart                      0.0859           
before                     0.0747           
surgery                    0.0538           
.                          0.0001           
[SEP]           

## koELECTRA로 한국어 실습

- KoELECTRA discriminator는 한국어 ELECTRA의 판별기 모델로 공개.

- ELECTRA 계열의 핵심은 generator가 만든 대체 토큰을 discriminator가 원래 토큰인지 아닌지 판별하는 RTD

- “빈칸 채우기”가 아니라 “수상한 토큰 찾기"

In [ ]:
# 모델 불러오기
# KoELECTRA는 genertor와 discriminator를 따로 불러올 수 있음.
import torch
from transformers import (
    AutoTokenizer,
    ElectraForPreTraining,
    ElectraForMaskedLM
)

disc_name = "monologg/koelectra-base-v3-discriminator"
gen_name  = "monologg/koelectra-base-v3-generator"

tokenizer = AutoTokenizer.from_pretrained(disc_name)
discriminator = ElectraForPreTraining.from_pretrained(disc_name)
generator = ElectraForMaskedLM.from_pretrained(gen_name)

discriminator.eval()
generator.eval()

print("로드 완료")

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/463 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/149M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

ElectraForMaskedLM LOAD REPORT from: monologg/koelectra-base-v3-generator
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


로드 완료


In [ ]:
# KoELECTRA의 핵심 출력 확인
## discriminator는 문장 각 토큰 위치 마다 "이 토큰이 원래 토큰이 나리 가능성"을 나타내는 logit을 나타냄

def koelectra_token_fake_scores(text):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = discriminator(**inputs)

    logits = outputs.logits[0]          # (seq_len,)
    probs_fake = torch.sigmoid(logits)  # fake score처럼 해석

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    results = []
    for tok, score in zip(tokens, probs_fake):
        results.append((tok, float(score)))
    return results

In [ ]:
# 정상 문장 VS 이상 문장 비교

normal_text = "의사는 수술 전에 환자의 차트를 확인했다."
odd_text    = "의사는 수술 전에 환자의 바나나를 확인했다."

for text in [normal_text, odd_text]:
    print("=" * 90)
    print("입력 문장:", text)
    print("-" * 90)

    scores = koelectra_token_fake_scores(text)
    for tok, score in scores:
        print(f"{tok:15s}  fake_score={score:.4f}")
    print()

입력 문장: 의사는 수술 전에 환자의 차트를 확인했다.
------------------------------------------------------------------------------------------
[CLS]            fake_score=0.0000
의사               fake_score=0.0294
##는              fake_score=0.0055
수술               fake_score=0.0293
전                fake_score=0.0182
##에              fake_score=0.0256
환자               fake_score=0.0480
##의              fake_score=0.0368
차트               fake_score=0.0232
##를              fake_score=0.0047
확인               fake_score=0.0420
##했              fake_score=0.0017
##다              fake_score=0.0017
.                fake_score=0.0002
[SEP]            fake_score=0.0000

입력 문장: 의사는 수술 전에 환자의 바나나를 확인했다.
------------------------------------------------------------------------------------------
[CLS]            fake_score=0.0000
의사               fake_score=0.0345
##는              fake_score=0.0047
수술               fake_score=0.0393
전                fake_score=0.0229
##에              fake_score=0.0200
환자               fa

- "차트"가 들어간 정상 문장에서는 대부분의 토큰 점수가 비교적 낮게 나올 가능성이 큼.

- "바나나"처럼 문맥상 이상한 단어가 들어가면 그 주변 토큰이나 해당 토큰의 fake score가 더 높아질 수 있음.

In [ ]:
def show_koelectra_scores(text, threshold=0.5):
    scores = koelectra_token_fake_scores(text)

    print("=" * 100)
    print("입력:", text)
    print("=" * 100)
    print(f"{'TOKEN':20s} {'FAKE_SCORE':>12s} {'FLAG':>10s}")
    print("-" * 100)

    for tok, score in scores:
        flag = "fake?" if score >= threshold else ""
        print(f"{tok:20s} {score:12.4f} {flag:>10s}")

show_koelectra_scores("대한민국의 수도는 서울이다.")
show_koelectra_scores("대한민국의 수도는 부산이다.")
show_koelectra_scores("의사는 수술 전에 환자의 바나나를 확인했다.")

입력: 대한민국의 수도는 서울이다.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000           
대한민국                       0.0092           
##의                        0.0017           
수도                         0.0050           
##는                        0.0042           
서울                         0.0095           
##이다                       0.0028           
.                          0.0005           
[SEP]                      0.0000           
입력: 대한민국의 수도는 부산이다.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000           
대한민국                       0.0815           
##의                        0.0070           
수도                         0.0303           
##는                        0.0069           
부산                         0.1820     

In [ ]:
#추천 관찰 문장

examples = [
    "대한민국의 수도는 서울이다.",
    "대한민국의 수도는 부산이다.",
    "이 논문의 핵심 기여는 새로운 최적화 기법에 있다.",
    "이 논문의 핵심 기여는 새로운 바나나 기법에 있다.",
    "의사는 수술 전에 환자의 차트를 확인했다.",
    "의사는 수술 전에 환자의 바나나를 확인했다."
]

for sent in examples:
    show_koelectra_scores(sent)
    print()

입력: 대한민국의 수도는 서울이다.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000           
대한민국                       0.0092           
##의                        0.0017           
수도                         0.0050           
##는                        0.0042           
서울                         0.0095           
##이다                       0.0028           
.                          0.0005           
[SEP]                      0.0000           

입력: 대한민국의 수도는 부산이다.
TOKEN                  FAKE_SCORE       FLAG
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000           
대한민국                       0.0815           
##의                        0.0070           
수도                         0.0303           
##는                        0.0069           
부산                         0.1820    

In [ ]:
# generator와 discriminator를 함께 보는 실습
## generator가 [MASK]를 채우고, discriminator가 그 결과를 다시 검사

def koelectra_full_demo(masked_text):
    print("=" * 100)
    print("원본 입력:", masked_text)

    # 1) generator로 [MASK] 채우기
    inputs = tokenizer(masked_text, return_tensors="pt")
    mask_id = tokenizer.mask_token_id
    mask_positions = (inputs["input_ids"][0] == mask_id).nonzero(as_tuple=True)[0]

    with torch.no_grad():
        gen_outputs = generator(**inputs)

    gen_logits = gen_outputs.logits[0]
    replaced_ids = inputs["input_ids"][0].clone()

    for pos in mask_positions:
        pred_id = gen_logits[pos].argmax().item()
        replaced_ids[pos] = pred_id

    replaced_text = tokenizer.decode(replaced_ids, skip_special_tokens=True)
    print("generator가 채운 문장:", replaced_text)

    # 2) discriminator로 fake score 계산
    disc_inputs = {
        "input_ids": replaced_ids.unsqueeze(0),
        "attention_mask": inputs["attention_mask"]
    }

    with torch.no_grad():
        disc_outputs = discriminator(**disc_inputs)

    disc_logits = disc_outputs.logits[0]
    disc_probs = torch.sigmoid(disc_logits)
    disc_tokens = tokenizer.convert_ids_to_tokens(replaced_ids)

    print("-" * 100)
    print(f"{'TOKEN':20s} {'FAKE_SCORE':>12s}")
    print("-" * 100)
    for tok, score in zip(disc_tokens, disc_probs):
        print(f"{tok:20s} {float(score):12.4f}")

In [ ]:
koelectra_full_demo("대한민국의 수도는 [MASK]이다.")
koelectra_full_demo("의사는 수술 전에 환자의 [MASK]를 확인했다.")
koelectra_full_demo("이 논문의 핵심 기여는 새로운 [MASK] 기법에 있다.")

원본 입력: 대한민국의 수도는 [MASK]이다.
generator가 채운 문장: 대한민국의 수도는 대한민국 이다.
----------------------------------------------------------------------------------------------------
TOKEN                  FAKE_SCORE
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000
대한민국                       0.0335
##의                        0.0068
수도                         0.2990
##는                        0.0084
대한민국                       0.6666
이                          0.0543
##다                        0.0074
.                          0.0008
[SEP]                      0.0000
원본 입력: 의사는 수술 전에 환자의 [MASK]를 확인했다.
generator가 채운 문장: 의사는 수술 전에 환자의 상태 를 확인했다.
----------------------------------------------------------------------------------------------------
TOKEN                  FAKE_SCORE
----------------------------------------------------------------------------------------------------
[CLS]                      0.0000
의사          

## 1. DeBERTa란?

DeBERTa는 **Decoding-enhanced BERT with Disentangled Attention**의 약자이며,  
BERT와 RoBERTa를 개선하기 위해 제안된 **Transformer encoder 기반 사전학습 언어모델**.

DeBERTa의 핵심 아이디어는 크게 두 가지.

1. **content 정보와 position 정보를 분리해서 처리하는 disentangled attention**
2. masked language modeling에서 **enhanced mask decoder**를 사용해 예측을 더 정교하게 만드는 방식


> **DeBERTa는 단어의 의미 정보와 위치 정보를 섞지 않고 분리해서 다루며, 마스크 토큰 복원도 더 정교하게 수행하도록 개선한 BERT 계열 모델이다.**

## 2. 왜 DeBERTa가 등장했는가?

BERT 계열 모델은 입력 토큰의 표현을 보통 다음처럼 만든다.

$$
\text{token embedding} + \text{position embedding}
$$

**단어의 내용(content)** 과 **위치(position)** 를 처음부터 더해서 하나의 벡터로 사용.

DeBERTa는 이 방식에 문제의식을 가진다.  
왜냐하면 다음 두 정보는 성격이 다르기 때문.

- **무엇이 쓰였는가**: 단어의 의미
- **어디에 쓰였는가**: 단어의 위치

DeBERTa는 이 둘을 너무 일찍 섞지 말고, **분리해서 attention 계산에 반영하면 더 좋은 문맥 표현을 얻을 수 있다**.

또한 masked language modeling에서도 단순히 hidden state만으로 복원하지 않고,  
**absolute position 정보까지 decoding 단계에서 활용하는 enhanced mask decoder**를 도입.


## 3. 핵심 아이디어

DeBERTa의 핵심은 크게 세 가지로 정리할 수 있다.

### 3.1 Disentangled Attention

기존 BERT 계열 모델은 각 단어를 하나의 hidden vector로 보고 attention을 계산한다.  
하지만 DeBERTa는 각 단어를 다음 두 표현으로 분리한다.

- **content vector**
- **position vector**

그리고 attention score를 계산할 때도 단순히 단어 벡터끼리만 비교하지 않고,  
**content와 relative position의 관계를 분리해서 반영**.

DeBERTa는 단순히

> “어떤 단어가 어떤 단어를 참고할까?”

만 묻지 않고,

> “이 단어의 의미가, 상대적 위치를 고려했을 때 다른 단어와 어떤 관계를 가지는가?”

를 더 명시적으로 반영.

---

### 3.2 Relative Position 사용 강화

BERT는 absolute position embedding을 입력에 더하는 방식에 가깝다.  

“몇 번째 위치인가”를 직접 입력에 포함한다.

반면 DeBERTa는 attention 계산에서  
**현재 단어와 다른 단어 사이의 상대적 거리(relative position)** 를 더 적극적으로 활용한다.

- 단순히 5번째 단어인지
- 8번째 단어인지

보다도,

- 현재 단어와 몇 칸 떨어져 있는지
- 앞에 있는지 뒤에 있는지

같은 관계 정보를 더 직접적으로 반영.

---

### 3.3 Enhanced Mask Decoder

DeBERTa는 masked language modeling을 수행할 때, 단순히 encoder의 마지막 hidden state만 사용하는 것이 아니라  
**absolute position 정보까지 decoding 단계에서 활용**.

masked token을 예측할 때

- 문맥에서 얻은 의미 정보
- 위치 정보

를 더 정교하게 결합하여 예측 성능을 높임.

이것이 이름의 **Decoding-enhanced** 부분.

## 4. BERT와 무엇이 다른가?

### 4.1 입력 표현 방식

#### BERT

BERT는 각 위치의 입력을 보통 다음처럼 만든다.

$$
h_i = E_i + P_i
$$

여기서

- $E_i$: 단어 임베딩
- $P_i$: 위치 임베딩

**내용과 위치를 처음부터 더해서 하나의 벡터로 사용**.

---

#### DeBERTa

DeBERTa는 내용과 위치를 분리해서 유지.


- 단어의 의미를 담는 **content representation**
- 위치 정보를 담는 **position representation**

을 따로 두고 attention 계산에서도 이를 별도로 활용.

이 차이가 매우 중요.

- **BERT**: “무엇”과 “어디”를 합쳐서 처리
- **DeBERTa**: “무엇”과 “어디”를 **분리해서 더 세밀하게 처리**

---

### 4.2 Attention 계산 방식

BERT의 self-attention은 주로  
content와 position이 합쳐진 hidden state 사이의 관계를 본다.

반면 DeBERTa는 attention에서 다음과 같은 관계를 분리해서 계산.

- **content-to-content**
- **content-to-position**

단순한 의미 유사도만 보는 것이 아니라 단어 내용과 상대 위치가 함께 주는 영향을 구조적으로 분리해서 반영한다.

---

### 4.3 Decoder 차이

BERT의 MLM은 masked position에 대해  
최종 hidden state로부터 softmax를 적용하는 구조에 가깝다.

반면 DeBERTa는 masked token을 예측할 때  
**position 정보까지 더 잘 활용하도록 decoder를 강화**.

이것이 **enhanced mask decoder**.

## 5. 수식으로 이해하기

### 5.1 BERT의 기본 입력

BERT에서는 보통 각 토큰의 입력을 다음처럼 둔다.

$$
h_i^{(0)} = E(x_i) + P_i
$$

여기서

- $E(x_i)$: 토큰 $x_i$의 embedding
- $P_i$: 위치 $i$의 position embedding


---

### 5.2 DeBERTa의 분리된 표현

DeBERTa는 각 단어를 두 종류의 벡터로 다룬다.

$$
c_i = \text{content embedding of token } x_i
$$

$$
p_i = \text{position embedding of position } i
$$

토큰의 의미 정보 $c_i$와 위치 정보 $p_i$를  
바로 합치지 않고 분리해서 유지.

---

### 5.3 일반적인 Self-Attention

일반적인 attention은 보통 다음과 같이 쓴다.

$$
\text{Attention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)V
$$

이 식은 query와 key의 유사도를 바탕으로  
각 value를 얼마나 참고할지를 결정한다.

---

### 5.4 DeBERTa의 Disentangled Attention 직관

DeBERTa에서는 attention score를 만들 때  단순히 hidden state끼리만 비교하는 것이 아니라,  
**content와 relative position을 분리한 항들**을 사용.

직관적으로는 다음처럼 이해할 수 있다.

$$
\text{score}_{ij}
=
\underbrace{c_i^\top W_{cc} c_j}_{\text{content-content}}
+
\underbrace{c_i^\top W_{cp} r_{ij}}_{\text{content-position}}
$$

여기서

- $c_i, c_j$: 토큰 $i, j$의 content 표현
- $r_{ij}$: $i$와 $j$ 사이의 relative position 표현
- $(W_{cc}, W_{cp})$: 학습 가능한 파라미터

attention score는 단순한 내용-내용 유사도만이 아니라 **내용과 상대 위치 관계까지 함께 반영**.

실제 논문 구현은 더 복잡하지만,  
핵심 직관은 다음 한 줄로 정리할 수 있다.

> **DeBERTa는 의미 관계와 위치 관계를 분리해서 attention에 넣는다.**

---

### 5.5 MLM Loss

DeBERTa도 기본적으로 MLM 기반으로 사전학습된다.  
마스크된 위치 집합을 \(M\)이라 하면 손실은 다음처럼 쓸 수 있다.

$$
\mathcal{L}_{MLM}
=
-\sum_{i \in M} \log p(x_i \mid x_{\setminus M})
$$

여기서 중요한 점은  
손실식 자체보다, **그 확률을 계산하는 encoder 표현과 decoder 구조가 달라졌다**는 것.

DeBERTa의 개선점은  
loss 함수의 형태보다는 **표현 방식과 attention 설계**에 있다.

---

## 6. DeBERTa의 장점

### 6.1 의미와 위치를 더 정교하게 구분

DeBERTa는 토큰의 의미 정보와 위치 정보를 섞지 않고 분리해서 다룬다.  
따라서 문장 안에서 **단어 간 관계를 더 세밀하게 포착**할 수 있다.

---

### 6.2 더 효율적인 표현 학습

단어 내용과 상대 위치를 분리하여 attention에 반영함으로써  
문맥 표현을 더 풍부하게 만들 수 있다.

같은 Transformer encoder 구조 안에서도 더 정교한 정보 사용이 가능하다.

---

### 6.3 강한 NLU 성능

DeBERTa는 특히 다음과 같은 **이해 중심 과제**에서 강하다.

- 문장 분류
- 자연어 추론
- 질의응답
- 독해
- 토큰 분류

GPT처럼 자유 생성보다는 **문장을 이해하고 판별하는 작업**에서 강점을 보임.


## 7. DeBERTa의 한계

### 7.1 구조가 더 복잡함

BERT는 “token embedding + position embedding”이라는 비교적 직관적인 구조이지만,  
DeBERTa는

- content / position 분리
- disentangled attention
- enhanced mask decoder

등이 들어가므로 설명과 구현이 더 복잡.

---

### 7.2 여전히 encoder 중심 모델

DeBERTa는 GPT처럼 긴 텍스트를 순차적으로 생성하는  
decoder-only 생성형 모델이 아니다.

기본적으로는 **encoder 기반 이해 모델**이다.

그래서 다음 같은 작업에는 매우 적합하지만,

- classification
- NLI
- QA
- tagging

다음 같은 **자유 생성 task**에는 직접 최적화된 구조는 아니다.

- 장문 생성
- 대화 생성
- 스토리 생성


## 8. DeBERTa-v2와 DeBERTa-v3

### 8.1 DeBERTa-v2

DeBERTa-v2에서는 추가적인 개선이 들어감.

대표적으로 알려진 특징은 다음과 같다.

- **SentencePiece 기반 tokenizer**
- 더 큰 vocabulary
- 첫 Transformer layer 안의 **추가 convolution layer**
- attention에서 **position/content projection 공유**로 파라미터 절감

원래 DeBERTa의 아이디어를 유지하면서 효율성과 구현 측면을 더 다듬은 버전이라고 볼 수 있다.

---

### 8.2 DeBERTa-v3

DeBERTa-v3는 DeBERTa 구조에 더해  
**ELECTRA-style pretraining** 아이디어를 결합한 버전으로 이해할 수 있다.


- DeBERTa의 **disentangled attention**
- ELECTRA의 **판별 중심 pretraining**

을 함께 사용하여 더 좋은 학습 효율과 성능을 노린다.ㅋ


## 9. BERT, RoBERTa, ELECTRA, DeBERTa 비교

| 항목 | BERT | RoBERTa | ELECTRA | DeBERTa |
|---|---|---|---|---|
| 기본 구조 | Transformer Encoder | Transformer Encoder | Transformer Encoder | Transformer Encoder |
| 입력 표현 | content + position를 합침 | BERT와 유사 | BERT와 유사 | content와 position를 분리 |
| 핵심 학습 목표 | MLM + NSP | MLM | RTD + 보조적 MLM | MLM + enhanced mask decoder |
| 핵심 개선점 | 기본 bidirectional encoder | 학습 최적화 | 판별 중심 학습 | disentangled attention + decoder 강화 |
| 위치 정보 처리 | absolute position 중심 | BERT와 유사 | BERT와 유사 | relative position을 더 적극 활용 |
| 강점 | 기본 구조 이해에 적합 | 더 강한 학습 버전 | 계산 효율성 | 더 정교한 의미-위치 분리 |

이 표의 핵심은 다음과 같다.

- **BERT**: 기본 encoder 모델
- **RoBERTa**: BERT를 더 잘 학습시킨 모델
- **ELECTRA**: 학습 목표 자체를 바꿔 효율을 높인 모델
- **DeBERTa**: 의미와 위치를 구조적으로 더 잘 분리한 모델

DeBERTa의 핵심은

> **“BERT를 더 오래 학습한 것”이 아니라, “attention에서 의미와 위치를 더 정교하게 분리한 것”**


In [ ]:
# DeBERTa 불러오기와 hidden state 확인

# %pip install -q transformers torch

import torch
from transformers import AutoTokenizer, AutoModel

model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "The patient underwent surgery yesterday."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state

print("input_ids shape :", inputs["input_ids"].shape)
print("last_hidden_state shape :", last_hidden_state.shape)


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


input_ids shape : torch.Size([1, 6])
last_hidden_state shape : torch.Size([1, 6, 768])


- last_hidden_state shape는 보통 (batch_size, seq_len, hidden_size)
- DeBERTa-V3-base의 hidden size는 768

In [ ]:
# 토큰화 결과 보기
## DeBERTa-v2계열과 V3계열은 SentencePiece 기반 토크나이저와 큰 vocabulary를 사용함.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

text = "The patient's preoperative assessment was incomplete."
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:")
print(tokens)
print()

print("Token IDs:")
print(ids)

Tokens:
['▁The', '▁patient', "'", 's', '▁preoperative', '▁assessment', '▁was', '▁incomplete', '.']

Token IDs:
[279, 1799, 280, 268, 56123, 3186, 284, 11818, 260]


- BERT와 토큰이 조금 다르게 잘릴 수 있음.
- vocabulary와 토크나이저 체계가 다르기 때문에 같은 문장도 토큰 분할이 다르게 나올 수 있음.

In [ ]:
# 단어 순서 변화에 얼마나 민감한지 보기
## DeBERTa는 content와 position을 분리해 attention에 반영하므로, 단어는 같아도 순서가 바뀌면 표현이 달라지는 정도를 확인해 보면 좋음
## DeBERTa의 핵심은 context와 position을 섞지 않고 처리하는데 있음

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # 평균 pooling
    emb = outputs.last_hidden_state.mean(dim=1)
    return emb

s1 = "The dog chased the cat."
s2 = "The cat chased the dog."
s3 = "The dog chased the cat in the yard."

e1 = sentence_embedding(s1)
e2 = sentence_embedding(s2)
e3 = sentence_embedding(s3)

sim12 = F.cosine_similarity(e1, e2).item()
sim13 = F.cosine_similarity(e1, e3).item()

print(f"Similarity(s1, s2) = {sim12:.4f}")
print(f"Similarity(s1, s3) = {sim13:.4f}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity(s1, s2) = 0.4824
Similarity(s1, s3) = 0.8359


- s1과 s2는 단어는 비슷하지만 의미 관계와 순서가 다름.
- s1과 s3는 더 길지만 기본 사건 구조는 유사.

- 이 차이를 통해 DeBERTa가 상대 위치와 문맥 관계를 어떻게 반영하는지 간접적으로 볼 수 있음

In [ ]:
# BERT와 DeBERTa를 같은 문장으로 비교
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

bert_name = "google-bert/bert-base-uncased"
deberta_name = "microsoft/deberta-v3-base"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name)
bert_model.eval()

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)
deberta_model = AutoModel.from_pretrained(deberta_name)
deberta_model.eval()

def get_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1)

pairs = [
    ("The dog chased the cat.", "The cat chased the dog."),
    ("The patient received treatment.", "The patient received medical treatment."),
    ("The capital of France is Paris.", "The capital of France is Lyon.")
]

for a, b in pairs:
    bert_a = get_embedding(a, bert_tokenizer, bert_model)
    bert_b = get_embedding(b, bert_tokenizer, bert_model)
    deberta_a = get_embedding(a, deberta_tokenizer, deberta_model)
    deberta_b = get_embedding(b, deberta_tokenizer, deberta_model)

    bert_sim = F.cosine_similarity(bert_a, bert_b).item()
    deberta_sim = F.cosine_similarity(deberta_a, deberta_b).item()

    print("=" * 80)
    print("A:", a)
    print("B:", b)
    print(f"BERT similarity    : {bert_sim:.4f}")
    print(f"DeBERTa similarity : {deberta_sim:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A: The dog chased the cat.
B: The cat chased the dog.
BERT similarity    : 0.9964
DeBERTa similarity : 0.4824
A: The patient received treatment.
B: The patient received medical treatment.
BERT similarity    : 0.9426
DeBERTa similarity : 0.8247
A: The capital of France is Paris.
B: The capital of France is Lyon.
BERT similarity    : 0.9520
DeBERTa similarity : 0.9189


In [ ]:
# Masked LM 실습
## DeBERTa는 enhanced mask decoder를 사용해 masked token prediction을 개선한 모델


import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = "microsoft/deberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

text = "The capital of France is [MASK]."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
mask_token_id = tokenizer.mask_token_id
mask_pos = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=True)[1]

mask_logits = logits[0, mask_pos, :]
top_k = 5
top_ids = torch.topk(mask_logits, top_k, dim=-1).indices[0].tolist()

print("Top predictions:")
for idx in top_ids:
    print(tokenizer.decode([idx]).strip())

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/559M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
DebertaForMaskedLM LOAD REPORT from: microsoft/deberta-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
cls.predictions.decoder.bias               | MISSING    | 
cls.predictions.transform.dense.weight     | MISSING    | 
cls.predictions.transform.LayerNorm.bias   | MISSING    | 
cls.predictions.bias                       | MISSING    | 
cls.predictions.transform.LayerNorm.weight

Top predictions:
Mandal
avoid
Govern
Var
oso


In [ ]:
# 간단한 문장 분류 fine-tuning틀
## DeBERTa는 생성형 모델보다 분류, 추론, 질의응답 같은 NLU 작업에 특히 강한 encoder 모델

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

data = {
    "text": [
        "I love this paper.",
        "This result is disappointing.",
        "The method is very effective.",
        "The experiment failed badly."
    ],
    "label": [1, 0, 1, 0]
}

dataset = Dataset.from_dict(data)

model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

encoded_dataset = dataset.map(preprocess)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./deberta-cls",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset
)

trainer.train()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias        

Step,Training Loss
1,0.830451
2,0.650921
3,291.500000
4,4150.000000
5,1697.500000
6,0.000000


TrainOutput(global_step=6, training_loss=1023.4135619799296, metrics={'train_runtime': 2.512, 'train_samples_per_second': 4.777, 'train_steps_per_second': 2.389, 'total_flos': 394673660928.0, 'train_loss': 1023.4135619799296, 'epoch': 3.0})

- 한국어로 진행

- lighthouse/mdeberta-v3-base-kor-further: 한국어 데이터로 추가 사전학습된 mDeBERTa-v3 기반 모델이라, 문장 표현 비교와 유사도 실습에 쓰기 좋음. 모델 카드는 이 모델이 microsoft/mDeBERTa-v3-base를 약 40GB 한국어 데이터에 대해 추가 사전학습한 모델이라고 설명.

- team-lucid/deberta-v3-base-korean: 한국어 NLU 성능표와 함께 공개된 한국어 DeBERTa-v3 계열 모델이라, 분류 실습에 쓰기 좋음. 모델 카드는 DeBERTa V3가 ELECTRA-style pretraining을 적용한 개선 버전이라고 설명.

In [ ]:
# 한국어 DeBERTa 모델 불러오기

import torch
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

model_name = "lighthouse/mdeberta-v3-base-kor-further"

tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name)
encoder.eval()

print("모델 로드 완료")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: lighthouse/mdeberta-v3-base-kor-further
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 로드 완료
cuda


In [ ]:
# 한국어 문장을 넣어서 hidden state 확인

text = "의사는 수술 전에 환자의 상태를 확인했다."

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = encoder(**inputs)

last_hidden_state = outputs.last_hidden_state

print("input_ids shape:", inputs["input_ids"].shape)
print("last_hidden_state shape:", last_hidden_state.shape)

input_ids shape: torch.Size([1, 15])
last_hidden_state shape: torch.Size([1, 15, 768])


- input_ids shape: 토큰 개수
- last_hidden_state shape: 각 토큰마다 몇 차원 벡터가 나오는지

- DeBERTa는 문장을 한 번에 하나의 벡터로 보는 것이 아니라,
각 토큰마다 문맥이 반영된 표현을 만듦.
    - lighthouse/mdeberta-v3-base-kor-further는 12 layers, hidden size 768 구조를 사용한다고 모델 카드

In [ ]:
# 한국어 토큰화 결과 보기
text = "이 논문의 핵심 기여는 새로운 최적화 기법을 제안한 것이다."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("토큰 목록:")
print(tokens)
print()

print("토큰 ID:")
print(token_ids)

토큰 목록:
['▁이', '▁논', '문의', '▁핵', '심', '▁기', '여', '는', '▁새', '로', '운', '▁최', '적', '화', '▁기', '법', '을', '▁', '제안', '한', '▁것이다', '.']

토큰 ID:
[1603, 38990, 59946, 72407, 8000, 6075, 7008, 989, 10207, 1236, 4192, 7045, 2504, 3258, 6075, 8281, 612, 260, 88225, 840, 19652, 261]


- DeBERTa 계열은 SentencePiece 기반 토크나이저를 사용할 수 있어서,
한국어 문장이 어절 단위 그대로가 아니라 서브워드 단위로 잘릴 수 있음.

- lighthouse/mdeberta-v3-base-kor-further 모델 카드는 SentencePiece 토크나이저를 사용한다고 설명

In [ ]:
# 한국어 문장 임베딩 만들기
## 가장 간단하게 평균 풀링을 사용함.

def 문장임베딩(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = encoder(**inputs)
    emb = outputs.last_hidden_state.mean(dim=1)
    return emb

In [ ]:
import torch.nn.functional as F
# 한국어 문장 유사도 비교
문장1 = "고양이가 쥐를 잡았다."
문장2 = "쥐가 고양이를 잡았다."
문장3 = "고양이가 어제 쥐를 잡았다."
문장4 = "학생이 시험 전에 개념을 복습했다."

emb1 = 문장임베딩(문장1)
emb2 = 문장임베딩(문장2)
emb3 = 문장임베딩(문장3)
emb4 = 문장임베딩(문장4)

sim12 = F.cosine_similarity(emb1, emb2).item()
sim13 = F.cosine_similarity(emb1, emb3).item()
sim14 = F.cosine_similarity(emb1, emb4).item()

print(f"문장1 vs 문장2 유사도: {sim12:.4f}")
print(f"문장1 vs 문장3 유사도: {sim13:.4f}")
print(f"문장1 vs 문장4 유사도: {sim14:.4f}")

문장1 vs 문장2 유사도: 0.8423
문장1 vs 문장3 유사도: 0.8911
문장1 vs 문장4 유사도: 0.7822


- 문장1 / 문장2: 단어는 비슷하지만 주어와 목적어 관계가 바뀜
- 문장1 / 문장3: 기본 사건은 같고 시간 정보만 추가됨
- 문장1 / 문장4: 아예 다른 의미

In [ ]:
# 여러 한국어 문장을 한 번에 비교하기

문장목록 = [
    "대한민국의 수도는 서울이다.",
    "대한민국의 수도는 부산이다.",
    "의사는 수술 전에 환자의 차트를 확인했다.",
    "의사는 수술 전에 환자의 바나나를 확인했다.",
    "학생은 시험 전에 개념을 복습했다.",
    "학생은 시험 전에 냉장고를 복습했다."
]

임베딩목록 = [문장임베딩(s) for s in 문장목록]

for i in range(len(문장목록)):
    for j in range(i + 1, len(문장목록)):
        sim = F.cosine_similarity(임베딩목록[i], 임베딩목록[j]).item()
        print("=" * 80)
        print("문장 A:", 문장목록[i])
        print("문장 B:", 문장목록[j])
        print(f"유사도: {sim:.4f}")

문장 A: 대한민국의 수도는 서울이다.
문장 B: 대한민국의 수도는 부산이다.
유사도: 0.9692
문장 A: 대한민국의 수도는 서울이다.
문장 B: 의사는 수술 전에 환자의 차트를 확인했다.
유사도: 0.6562
문장 A: 대한민국의 수도는 서울이다.
문장 B: 의사는 수술 전에 환자의 바나나를 확인했다.
유사도: 0.6675
문장 A: 대한민국의 수도는 서울이다.
문장 B: 학생은 시험 전에 개념을 복습했다.
유사도: 0.7153
문장 A: 대한민국의 수도는 서울이다.
문장 B: 학생은 시험 전에 냉장고를 복습했다.
유사도: 0.7529
문장 A: 대한민국의 수도는 부산이다.
문장 B: 의사는 수술 전에 환자의 차트를 확인했다.
유사도: 0.6558
문장 A: 대한민국의 수도는 부산이다.
문장 B: 의사는 수술 전에 환자의 바나나를 확인했다.
유사도: 0.6602
문장 A: 대한민국의 수도는 부산이다.
문장 B: 학생은 시험 전에 개념을 복습했다.
유사도: 0.6978
문장 A: 대한민국의 수도는 부산이다.
문장 B: 학생은 시험 전에 냉장고를 복습했다.
유사도: 0.7363
문장 A: 의사는 수술 전에 환자의 차트를 확인했다.
문장 B: 의사는 수술 전에 환자의 바나나를 확인했다.
유사도: 0.9419
문장 A: 의사는 수술 전에 환자의 차트를 확인했다.
문장 B: 학생은 시험 전에 개념을 복습했다.
유사도: 0.8521
문장 A: 의사는 수술 전에 환자의 차트를 확인했다.
문장 B: 학생은 시험 전에 냉장고를 복습했다.
유사도: 0.8281
문장 A: 의사는 수술 전에 환자의 바나나를 확인했다.
문장 B: 학생은 시험 전에 개념을 복습했다.
유사도: 0.8770
문장 A: 의사는 수술 전에 환자의 바나나를 확인했다.
문장 B: 학생은 시험 전에 냉장고를 복습했다.
유사도: 0.8643
문장 A: 학생은 시험 전에 개념을 복습했다.
문장 B: 학생은 시험 전에 냉장고를 복습했다.
유사도: 0.9302


In [ ]:
# 작은 한국어 분류 데이터 만들기

data = {
    "text": [
        "이 강의는 이해하기 쉽고 좋았다.",
        "설명이 너무 복잡해서 따라가기 힘들었다.",
        "실습이 매우 도움이 되었다.",
        "예제가 부족해서 아쉬웠다.",
        "교재 구성이 체계적이었다.",
        "수업 진행이 너무 산만했다.",
    ],
    "label": [1, 0, 1, 0, 1, 0],
}

dataset = Dataset.from_dict(data)
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 6
})

In [ ]:
# 전처리 함수
def 전처리(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64,
    )
## 데이터 토큰화

encoded_dataset = dataset.map(전처리)
encoded_dataset


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6
})

In [ ]:
# 분류 모델 불러오기
cls_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)

print(type(cls_model))

cls_model = cls_model.to(device)
cls_model.eval()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: lighthouse/mdeberta-v3-base-kor-further
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<class 'transformers.models.deberta_v2.modeling_deberta_v2.DebertaV2ForSequenceClassification'>


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [ ]:
# 학습 설정 만들기
bf16가능 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir="./ko_deberta_cls",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
    fp16=False,
    bf16=bf16가능,
)
# hugging Face 문서도 fp16은 overflow 위험이 있고, bf16은 더 넓은 수치 범위를 제공한다고 설명
## 학습이 float16에서 터져서 모델 출력이 NaN으로 붕괴한 상태

In [ ]:
# Trainer 만들기
trainer = Trainer(
    model=cls_model,
    args=training_args,
    train_dataset=encoded_dataset,
)

## 학습 시작
trainer.train()

Step,Training Loss
1,0.717935
2,0.738658
3,0.042734


TrainOutput(global_step=3, training_loss=0.499775721381108, metrics={'train_runtime': 1.0239, 'train_samples_per_second': 5.86, 'train_steps_per_second': 2.93, 'total_flos': 197336830464.0, 'train_loss': 0.499775721381108, 'epoch': 1.0})

In [ ]:
# 분류 파이프라인 만들기
classifier = pipeline(
    "text-classification",
    model=cls_model,
    tokenizer=tokenizer
)

In [ ]:
테스트배치 = tokenizer(
    [
    "설명이 친절하고 이해하기 쉬웠다.",
    "실습 예제가 너무 부족해서 아쉬웠다.",
    "전체적으로 매우 만족스러운 수업이었다.",
    "강의 속도가 너무 빨라서 따라가기 어려웠다."
    ],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

테스트배치 = {k: v.to(device) for k, v in 테스트배치.items()}

with torch.no_grad():
    출력 = cls_model(**테스트배치)

print("logits:")
print(출력.logits)

print("NaN 포함 여부:", torch.isnan(출력.logits).any().item())

for 문장 in 테스트배치:
    결과 = classifier(문장)
    print("=" * 80)
    print("입력 문장:", 문장)
    print("예측 결과:", 결과)

logits:
tensor([[ 0.2295, -0.5742],
        [ 9.2500, -9.9375],
        [ 2.1250, -2.5000],
        [ 0.1309, -0.8711]], device='cuda:0')
NaN 포함 여부: False
입력 문장: input_ids
예측 결과: [{'label': 'LABEL_1', 'score': 0.9999997615814209}]
입력 문장: token_type_ids
예측 결과: [{'label': 'LABEL_0', 'score': 0.9999251365661621}]
입력 문장: attention_mask
예측 결과: [{'label': 'LABEL_0', 'score': 0.9507778882980347}]


In [ ]:
# 라벨 해석하기

라벨설명 = {
    "LABEL_0": "부정",
    "LABEL_1": "긍정"
}

for 문장 in 테스트배치:
    결과 = classifier(문장)[0]
    print("=" * 80)
    print("입력 문장:", 문장)
    print("예측 라벨:", 결과["label"])
    print("해석:", 라벨설명[결과["label"]])
    print("점수:", round(결과["score"], 4))

입력 문장: input_ids
예측 라벨: LABEL_0
해석: 부정
점수: 0.9962
입력 문장: token_type_ids
예측 라벨: LABEL_1
해석: 긍정
점수: 1.0
입력 문장: attention_mask
예측 라벨: LABEL_0
해석: 부정
점수: 0.9852


In [ ]:
확률 = torch.softmax(출력.logits, dim=-1)
예측 = torch.argmax(확률, dim=-1)

라벨설명 = {0: "부정", 1: "긍정"}

for 문장, pred, prob in zip(테스트배치, 예측, 확률):
    print("=" * 80)
    print("입력 문장:", 문장)
    print("예측:", 라벨설명[int(pred)])
    print("확률:", prob.tolist())

입력 문장: input_ids
예측: 부정
확률: [0.6907677054405212, 0.30923229455947876]
입력 문장: token_type_ids
예측: 부정
확률: [1.0, 4.6448813684207835e-09]
입력 문장: attention_mask
예측: 부정
확률: [0.9902915358543396, 0.009708477184176445]
